# Poultry Feces Classification: Final Hierarchical Pipeline

This notebook implements the final hierarchical pipeline for poultry feces image classification.

## Problem formulation

The system is organized into three sequential stages:

1. **Stage 0**: `feces_present` vs `no_feces`
2. **Stage 1**: if feces are present, `healthy` vs `unhealthy`
3. **Stage 2**: if the sample is predicted as unhealthy, disease-specific classification into:
   - `coccidiosis`
   - `newcastle`
   - `salmonella`

## Objective

The main objective of this notebook is to build a clean, reproducible, and professionally structured implementation of the final hierarchical classification system.

This notebook is intended as a consolidated final pipeline rather than an exploratory workbench. Therefore, only essential checks are included, while extensive dataset auditing and exploratory diagnostics are intentionally omitted because they were already completed in previous development notebooks.

## General methodological rationale

The hierarchical decomposition follows a progressive decision logic:

- first, determine whether the image contains a usable fecal sample,
- then, distinguish normal from abnormal samples,
- and finally, classify the specific disease only within the pathological subset.

This design is consistent with the structure of the problem and helps separate three distinct decision layers: input validity, health status, and disease subtype.

# Model choice and training rationale

All image classification stages in this notebook are implemented using **ResNet-18** backbones initialized with ImageNet-pretrained weights.

Residual networks were introduced to ease optimization in deep architectures by learning residual mappings instead of directly learning unreferenced transformations. This formulation enabled stable training of substantially deeper convolutional networks and became a standard backbone family for visual recognition tasks. :contentReference[oaicite:0]{index=0}

In this notebook, the implementation relies on the official `torchvision.models.resnet18` model builder. The TorchVision documentation states that `ResNet18_Weights.DEFAULT` corresponds to the default ImageNet-pretrained weights provided for the model. :contentReference[oaicite:1]{index=1}

For optimization, the training setup uses:

- **CrossEntropyLoss**, which is the standard PyTorch criterion for multi-class classification from unnormalized logits. :contentReference[oaicite:2]{index=2}
- **Adam**, a widely used stochastic optimization method available in the official PyTorch optimizer API. :contentReference[oaicite:3]{index=3}

The goal here is not to propose a novel architecture, but to provide a robust and interpretable final implementation of the hierarchical pipeline.

# Reproducibility and implementation scope

This notebook includes:

- deterministic seed configuration where appropriate,
- explicit path management,
- stage-wise dataset definitions,
- model training and evaluation code,
- checkpoint saving,
- final hierarchical inference logic.

This notebook does not repeat the full exploratory validation of the dataset, duplicate analysis, or earlier experimental branches. Those steps were already completed in previous work and are intentionally kept outside this final implementation.

In [1]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

from torchvision import models, transforms

# Environment configuration

This section defines the core runtime configuration used throughout the notebook:

- random seed,
- computation device,
- project root,
- main input and output directories.

Only essential setup steps are included.

In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [3]:
ROOT = Path(r"C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\PII\chicken_feces_dataset")

NO_FECES_GATE_DIR = ROOT / "02_work" / "no_feces_gate"
NO_FECES_GATE_IMAGES_DIR = NO_FECES_GATE_DIR / "images"
NO_FECES_GATE_LABELS_CSV = NO_FECES_GATE_DIR / "labels.csv"

MULTICLASS_SPLITS_DIR = ROOT / "03_final" / "splits" / "rebuild_v1"
HIERARCHICAL_SPLITS_DIR = ROOT / "03_final" / "splits" / "rebuild_v1_hierarchical"

FINAL_OUTPUT_DIR = ROOT / "03_final" / "outputs_final_pipeline"
FINAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("FINAL_OUTPUT_DIR:", FINAL_OUTPUT_DIR)

ROOT: C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\PII\chicken_feces_dataset
FINAL_OUTPUT_DIR: C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\PII\chicken_feces_dataset\03_final\outputs_final_pipeline


# Essential path checks

The final notebook only performs minimal path validation to ensure that the required inputs for the three-stage pipeline are available before training begins.

In [4]:
required_paths = {
    "NO_FECES_GATE_LABELS_CSV": NO_FECES_GATE_LABELS_CSV,
    "MULTICLASS_TRAIN_CSV": MULTICLASS_SPLITS_DIR / "train.csv",
    "MULTICLASS_VAL_CSV": MULTICLASS_SPLITS_DIR / "val.csv",
    "MULTICLASS_TEST_CSV": MULTICLASS_SPLITS_DIR / "test.csv",
    "BINARY_TRAIN_CSV": HIERARCHICAL_SPLITS_DIR / "train_binary.csv",
    "BINARY_VAL_CSV": HIERARCHICAL_SPLITS_DIR / "val_binary.csv",
    "BINARY_TEST_CSV": HIERARCHICAL_SPLITS_DIR / "test_binary.csv",
    "DISEASE_TRAIN_CSV": HIERARCHICAL_SPLITS_DIR / "train_disease.csv",
    "DISEASE_VAL_CSV": HIERARCHICAL_SPLITS_DIR / "val_disease.csv",
    "DISEASE_TEST_CSV": HIERARCHICAL_SPLITS_DIR / "test_disease.csv",
}

missing_paths = []

for name, path in required_paths.items():
    exists = path.exists()
    print(f"{name}: {exists} -> {path}")
    if not exists:
        missing_paths.append((name, path))

if missing_paths:
    raise FileNotFoundError(
        "Some required files are missing:\n" +
        "\n".join([f"{name}: {path}" for name, path in missing_paths])
    )

NO_FECES_GATE_LABELS_CSV: True -> C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\PII\chicken_feces_dataset\02_work\no_feces_gate\labels.csv
MULTICLASS_TRAIN_CSV: True -> C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\PII\chicken_feces_dataset\03_final\splits\rebuild_v1\train.csv
MULTICLASS_VAL_CSV: True -> C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\PII\chicken_feces_dataset\03_final\splits\rebuild_v1\val.csv
MULTICLASS_TEST_CSV: True -> C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\PII\chicken_feces_dataset\03_final\splits\rebuild_v1\test.csv
BINARY_TRAIN_CSV: True -> C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\PII\chicken_feces_dataset\03_final\splits\rebuild_v1_hierarchical\train_binary.csv
BINARY_VAL_CSV: True -> C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\PII\chicken_feces_dataset\03_final\splits\rebuild_v1_hierarchical\val_binary.csv
BINARY_TEST_CSV: True -> C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\PII\chicken_feces_dataset\03_final\splits\rebuild_v1_hierarchical\test_bi

# Shared image preprocessing configuration

All stages in this notebook use a common input size of **224 × 224**, which matches the standard expected input resolution of ResNet-18 in TorchVision. :contentReference[oaicite:4]{index=4}

At this stage, only the core preprocessing pipelines are defined. Stage-specific training and evaluation transforms will be instantiated when each dataset is introduced.

In [6]:
IMAGE_SIZE = (224, 224)

eval_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor()
])

train_transform_basic = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor()
])

# Output organization

To keep the final implementation clean and reproducible, the outputs of this notebook will be stored by stage.

Separate subdirectories will be used for:

- Stage 0: feces presence gate
- Stage 1: healthy vs unhealthy
- Stage 2: disease classification
- Final hierarchical predictions

In [7]:
STAGE0_OUTPUT_DIR = FINAL_OUTPUT_DIR / "stage0_feces_presence"
STAGE1_OUTPUT_DIR = FINAL_OUTPUT_DIR / "stage1_binary_health"
STAGE2_OUTPUT_DIR = FINAL_OUTPUT_DIR / "stage2_disease_classifier"
PIPELINE_OUTPUT_DIR = FINAL_OUTPUT_DIR / "hierarchical_pipeline"

for out_dir in [STAGE0_OUTPUT_DIR, STAGE1_OUTPUT_DIR, STAGE2_OUTPUT_DIR, PIPELINE_OUTPUT_DIR]:
    out_dir.mkdir(parents=True, exist_ok=True)

print("Stage 0 output:", STAGE0_OUTPUT_DIR)
print("Stage 1 output:", STAGE1_OUTPUT_DIR)
print("Stage 2 output:", STAGE2_OUTPUT_DIR)
print("Pipeline output:", PIPELINE_OUTPUT_DIR)

Stage 0 output: C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\PII\chicken_feces_dataset\03_final\outputs_final_pipeline\stage0_feces_presence
Stage 1 output: C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\PII\chicken_feces_dataset\03_final\outputs_final_pipeline\stage1_binary_health
Stage 2 output: C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\PII\chicken_feces_dataset\03_final\outputs_final_pipeline\stage2_disease_classifier
Pipeline output: C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\PII\chicken_feces_dataset\03_final\outputs_final_pipeline\hierarchical_pipeline


# Next step

The next section introduces **Stage 0**, the auxiliary input gate that distinguishes between:

- `feces_present`
- `no_feces`

This stage is designed as a preliminary validity filter before the health-status classifier and the disease classifier.

# Stage 0 — Feces presence gate

The first stage of the hierarchical pipeline is an auxiliary binary classifier that determines whether the input image contains a visible fecal sample.

## Stage definition

The two classes for this stage are:

- `feces_present`
- `no_feces`

## Role within the full pipeline

This stage acts as a preliminary validity filter:

- if the image is predicted as `no_feces`, the pipeline stops and the sample is treated as a non-diagnostic input,
- if the image is predicted as `feces_present`, the image is forwarded to the health-status classifier.

## Scope of this stage

This auxiliary dataset is intentionally kept separate from the main feces dataset used for pathological classification. Its function is not to redefine the clinical problem, but to prevent the downstream classifiers from producing diagnostic predictions on inputs without a visible fecal sample.

# Stage 0 dataset loading and split definition

The standardized Stage 0 metadata file is now loaded and used to define the train, validation, and test partitions for the feces-presence gate.

A stratified split is used in order to preserve class balance across the three subsets:

- 70% training
- 15% validation
- 15% test

This split provides a more stable evaluation setting than the earlier proof-of-concept configuration.

In [13]:
STAGE0_DIR = ROOT / "02_work" / "stage0_feces_presence"
STAGE0_IMAGES_DIR = STAGE0_DIR / "images"
STAGE0_LABELS_CSV = STAGE0_DIR / "labels.csv"

stage0_df = pd.read_csv(STAGE0_LABELS_CSV).copy()

print("Stage 0 total samples:", len(stage0_df))
print(stage0_df["label"].value_counts())
stage0_df.head()

Stage 0 total samples: 200
label
feces_present    100
no_feces         100
Name: count, dtype: int64


,filename,label
0,feces_present_001.jpg,feces_present
1,feces_present_002.jpg,feces_present
2,feces_present_003.jpg,feces_present
3,feces_present_004.jpg,feces_present
4,feces_present_005.jpg,feces_present


In [14]:
def build_stage0_path(filename: str, label: str) -> Path:
    return STAGE0_IMAGES_DIR / label / filename

stage0_df["filepath"] = stage0_df.apply(
    lambda row: build_stage0_path(row["filename"], row["label"]),
    axis=1
)

stage0_df["exists"] = stage0_df["filepath"].apply(Path.exists)

if not stage0_df["exists"].all():
    missing_files = stage0_df.loc[~stage0_df["exists"], ["filename", "label", "filepath"]]
    raise FileNotFoundError(
        "Some Stage 0 files are missing:\n" + missing_files.to_string(index=False)
    )

print("All Stage 0 files were found successfully.")

FileNotFoundError: Some Stage 0 files are missing:
             filename         label                                                                                                                                         filepath
feces_present_096.jpg feces_present C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\PII\chicken_feces_dataset\02_work\stage0_feces_presence\images\feces_present\feces_present_096.jpg
feces_present_097.jpg feces_present C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\PII\chicken_feces_dataset\02_work\stage0_feces_presence\images\feces_present\feces_present_097.jpg
     no_feces_056.jpg      no_feces           C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\PII\chicken_feces_dataset\02_work\stage0_feces_presence\images\no_feces\no_feces_056.jpg
     no_feces_057.jpg      no_feces           C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\PII\chicken_feces_dataset\02_work\stage0_feces_presence\images\no_feces\no_feces_057.jpg
     no_feces_058.jpg      no_feces         

# Stage 0 label encoding

The binary label encoding for the feces-presence gate is defined as follows:

- `feces_present` -> `0`
- `no_feces` -> `1`

In [15]:
stage0_class_to_idx = {
    "feces_present": 0,
    "no_feces": 1
}

stage0_idx_to_class = {v: k for k, v in stage0_class_to_idx.items()}

stage0_df["label_idx"] = stage0_df["label"].map(stage0_class_to_idx)

display(stage0_df.head())
print(stage0_df["label_idx"].value_counts().sort_index())

,filename,label,filepath,exists,label_idx
0,feces_present_001.jpg,feces_present,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,True,0
1,feces_present_002.jpg,feces_present,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,True,0
2,feces_present_003.jpg,feces_present,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,True,0
3,feces_present_004.jpg,feces_present,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,True,0
4,feces_present_005.jpg,feces_present,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,True,0


label_idx
0    100
1    100
Name: count, dtype: int64


# Stage 0 stratified split

A two-step stratified split is applied:

1. train vs temporary holdout
2. validation vs test from the temporary holdout

This produces the final 70/15/15 partition.

In [16]:
stage0_train_df, stage0_temp_df = train_test_split(
    stage0_df,
    test_size=0.30,
    random_state=SEED,
    stratify=stage0_df["label"]
)

stage0_val_df, stage0_test_df = train_test_split(
    stage0_temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=stage0_temp_df["label"]
)

print("Stage 0 split sizes:")
print("  train:", len(stage0_train_df))
print("  val:  ", len(stage0_val_df))
print("  test: ", len(stage0_test_df))

print("\nTrain distribution:")
print(stage0_train_df["label"].value_counts())

print("\nValidation distribution:")
print(stage0_val_df["label"].value_counts())

print("\nTest distribution:")
print(stage0_test_df["label"].value_counts())

Stage 0 split sizes:
  train: 140
  val:   30
  test:  30

Train distribution:
label
feces_present    70
no_feces         70
Name: count, dtype: int64

Validation distribution:
label
feces_present    15
no_feces         15
Name: count, dtype: int64

Test distribution:
label
feces_present    15
no_feces         15
Name: count, dtype: int64


# Stage 0 image preprocessing

The feces-presence gate uses the same input resolution as the rest of the pipeline.

A lightweight augmentation setup is used for training, while validation and test use deterministic preprocessing only.

In [17]:
stage0_train_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor()
])

stage0_eval_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor()
])

# Stage 0 dataset class

A dedicated PyTorch dataset is defined for the feces-presence gate.

The dataset returns:

- image tensor
- numeric label
- filename
- resolved filepath

In [18]:
class Stage0Dataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None):
        self.df = df.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = Path(row["filepath"])

        try:
            image = Image.open(image_path).convert("RGB")
        except Exception as e:
            raise RuntimeError(f"Error opening Stage 0 image: {image_path} | {e}")

        label = int(row["label_idx"])

        if self.transform is not None:
            image = self.transform(image)

        return image, label, row["filename"], str(image_path)

# Stage 0 datasets and dataloaders

The train, validation, and test subsets are instantiated as PyTorch datasets and wrapped into dataloaders.

In [19]:
stage0_train_dataset = Stage0Dataset(stage0_train_df, transform=stage0_train_transform)
stage0_val_dataset = Stage0Dataset(stage0_val_df, transform=stage0_eval_transform)
stage0_test_dataset = Stage0Dataset(stage0_test_df, transform=stage0_eval_transform)

stage0_train_loader = DataLoader(stage0_train_dataset, batch_size=16, shuffle=True)
stage0_val_loader = DataLoader(stage0_val_dataset, batch_size=16, shuffle=False)
stage0_test_loader = DataLoader(stage0_test_dataset, batch_size=16, shuffle=False)

print("Stage 0 loaders created successfully.")
print("Train batches:", len(stage0_train_loader))
print("Val batches:  ", len(stage0_val_loader))
print("Test batches: ", len(stage0_test_loader))

Stage 0 loaders created successfully.
Train batches: 9
Val batches:   2
Test batches:  2


# Stage 0 model definition

The feces-presence gate is implemented with a pretrained ResNet-18 backbone.

The final fully connected layer is replaced with a two-output linear classifier corresponding to the two Stage 0 classes.

In [20]:
stage0_model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
stage0_in_features = stage0_model.fc.in_features
stage0_model.fc = nn.Linear(stage0_in_features, 2)
stage0_model = stage0_model.to(device)

stage0_criterion = nn.CrossEntropyLoss()
stage0_optimizer = torch.optim.Adam(stage0_model.parameters(), lr=1e-4)

print(stage0_model.fc)

Linear(in_features=512, out_features=2, bias=True)


# Stage 0 training and evaluation utilities

This section defines the helper functions required for:

- one training epoch,
- full-dataset prediction,
- metric computation,
- checkpoint selection based on validation macro F1.

In [21]:
def train_one_epoch_stage0(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    y_true = []
    y_pred = []

    for images, labels, _, _ in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(logits, dim=1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(y_true, y_pred)
    epoch_macro_f1 = f1_score(y_true, y_pred, average="macro")

    return epoch_loss, epoch_acc, epoch_macro_f1


@torch.no_grad()
def predict_stage0(model, loader, device):
    model.eval()
    rows = []

    for images, labels, filenames, filepaths in loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        for i in range(len(filenames)):
            true_idx = int(labels[i].cpu().item())
            pred_idx = int(preds[i].cpu().item())

            rows.append({
                "filename": filenames[i],
                "filepath": filepaths[i],
                "true_label": stage0_idx_to_class[true_idx],
                "pred_label": stage0_idx_to_class[pred_idx],
                "correct": true_idx == pred_idx,
                "confidence": float(probs[i].max().cpu().item()),
                "prob_feces_present": float(probs[i][stage0_class_to_idx["feces_present"]].cpu().item()),
                "prob_no_feces": float(probs[i][stage0_class_to_idx["no_feces"]].cpu().item())
            })

    return pd.DataFrame(rows)


def evaluate_stage0(model, loader, device):
    results_df = predict_stage0(model, loader, device=device)

    y_true = results_df["true_label"]
    y_pred = results_df["pred_label"]

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=["feces_present", "no_feces"]),
        "classification_report": classification_report(
            y_true,
            y_pred,
            labels=["feces_present", "no_feces"],
            zero_division=0
        )
    }

    return results_df, metrics

# Stage 0 output paths

Dedicated output paths are defined for:

- best checkpoint,
- training history,
- validation predictions,
- test predictions.

In [22]:
STAGE0_MODEL_PATH = STAGE0_OUTPUT_DIR / "best_stage0_feces_presence_resnet18.pt"
STAGE0_HISTORY_PATH = STAGE0_OUTPUT_DIR / "stage0_training_history.csv"
STAGE0_VAL_PREDS_PATH = STAGE0_OUTPUT_DIR / "stage0_val_predictions.csv"
STAGE0_TEST_PREDS_PATH = STAGE0_OUTPUT_DIR / "stage0_test_predictions.csv"

print("Stage 0 model path:", STAGE0_MODEL_PATH)

Stage 0 model path: C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\PII\chicken_feces_dataset\03_final\outputs_final_pipeline\stage0_feces_presence\best_stage0_feces_presence_resnet18.pt


# Stage 0 stratified split

A two-step stratified split is applied in order to preserve class balance across the train, validation, and test subsets.

The final partition is:

- 70% training
- 15% validation
- 15% test

This provides a stable experimental setup for the feces-presence gate.

In [31]:
stage0_class_to_idx = {
    "feces_present": 0,
    "no_feces": 1
}

stage0_idx_to_class = {v: k for k, v in stage0_class_to_idx.items()}

stage0_df["label_idx"] = stage0_df["label"].map(stage0_class_to_idx)

stage0_train_df, stage0_temp_df = train_test_split(
    stage0_df,
    test_size=0.30,
    random_state=SEED,
    stratify=stage0_df["label"]
)

stage0_val_df, stage0_test_df = train_test_split(
    stage0_temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=stage0_temp_df["label"]
)

print("Stage 0 split sizes:")
print("  train:", len(stage0_train_df))
print("  val:  ", len(stage0_val_df))
print("  test: ", len(stage0_test_df))

print("\nTrain distribution:")
print(stage0_train_df["label"].value_counts())

print("\nValidation distribution:")
print(stage0_val_df["label"].value_counts())

print("\nTest distribution:")
print(stage0_test_df["label"].value_counts())

Stage 0 split sizes:
  train: 140
  val:   30
  test:  30

Train distribution:
label
feces_present    70
no_feces         70
Name: count, dtype: int64

Validation distribution:
label
feces_present    15
no_feces         15
Name: count, dtype: int64

Test distribution:
label
feces_present    15
no_feces         15
Name: count, dtype: int64


# Stage 0 preprocessing

The feces-presence gate uses the same input resolution as the rest of the pipeline.

A lightweight augmentation setup is used for training, while validation and test rely on deterministic preprocessing only.

In [32]:
stage0_train_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor()
])

stage0_eval_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor()
])

# Stage 0 dataset class

A dedicated PyTorch dataset is defined for the feces-presence gate.

Each sample returns:

- image tensor
- numeric label
- filename
- resolved filepath

In [34]:
class Stage0Dataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None):
        self.df = df.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = Path(row["filepath"])

        try:
            image = Image.open(image_path).convert("RGB")
        except Exception as e:
            raise RuntimeError(f"Error opening Stage 0 image: {image_path} | {e}")

        label = int(row["label_idx"])

        if self.transform is not None:
            image = self.transform(image)

        return image, label, row["filename"], str(image_path)

# Stage 0 datasets and dataloaders

The train, validation, and test partitions are instantiated as PyTorch datasets and wrapped into dataloaders.

In [35]:
stage0_train_dataset = Stage0Dataset(stage0_train_df, transform=stage0_train_transform)
stage0_val_dataset = Stage0Dataset(stage0_val_df, transform=stage0_eval_transform)
stage0_test_dataset = Stage0Dataset(stage0_test_df, transform=stage0_eval_transform)

stage0_train_loader = DataLoader(stage0_train_dataset, batch_size=16, shuffle=True)
stage0_val_loader = DataLoader(stage0_val_dataset, batch_size=16, shuffle=False)
stage0_test_loader = DataLoader(stage0_test_dataset, batch_size=16, shuffle=False)

print("Stage 0 loaders created successfully.")
print("Train batches:", len(stage0_train_loader))
print("Val batches:  ", len(stage0_val_loader))
print("Test batches: ", len(stage0_test_loader))

Stage 0 loaders created successfully.
Train batches: 9
Val batches:   2
Test batches:  2


# Stage 0 model definition

The feces-presence gate is implemented with a pretrained ResNet-18 backbone.

The final fully connected layer is replaced with a two-output linear classifier corresponding to the two Stage 0 classes.

In [36]:
stage0_model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
stage0_in_features = stage0_model.fc.in_features
stage0_model.fc = nn.Linear(stage0_in_features, 2)
stage0_model = stage0_model.to(device)

stage0_criterion = nn.CrossEntropyLoss()
stage0_optimizer = torch.optim.Adam(stage0_model.parameters(), lr=1e-4)

print(stage0_model.fc)

Linear(in_features=512, out_features=2, bias=True)


# Stage 0 training and evaluation utilities

This section defines helper functions for:

- one training epoch,
- full-dataset prediction,
- metric computation,
- checkpoint selection based on validation macro F1.

In [37]:
def train_one_epoch_stage0(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    y_true = []
    y_pred = []

    for images, labels, _, _ in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(logits, dim=1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(y_true, y_pred)
    epoch_macro_f1 = f1_score(y_true, y_pred, average="macro")

    return epoch_loss, epoch_acc, epoch_macro_f1


@torch.no_grad()
def predict_stage0(model, loader, device):
    model.eval()
    rows = []

    for images, labels, filenames, filepaths in loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        for i in range(len(filenames)):
            true_idx = int(labels[i].cpu().item())
            pred_idx = int(preds[i].cpu().item())

            rows.append({
                "filename": filenames[i],
                "filepath": filepaths[i],
                "true_label": stage0_idx_to_class[true_idx],
                "pred_label": stage0_idx_to_class[pred_idx],
                "correct": true_idx == pred_idx,
                "confidence": float(probs[i].max().cpu().item()),
                "prob_feces_present": float(probs[i][stage0_class_to_idx["feces_present"]].cpu().item()),
                "prob_no_feces": float(probs[i][stage0_class_to_idx["no_feces"]].cpu().item())
            })

    return pd.DataFrame(rows)


def evaluate_stage0(model, loader, device):
    results_df = predict_stage0(model, loader, device=device)

    y_true = results_df["true_label"]
    y_pred = results_df["pred_label"]

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=["feces_present", "no_feces"]),
        "classification_report": classification_report(
            y_true,
            y_pred,
            labels=["feces_present", "no_feces"],
            zero_division=0
        )
    }

    return results_df, metrics

# Stage 0 output paths

Dedicated output paths are defined for:

- best checkpoint,
- training history,
- validation predictions,
- test predictions.

In [38]:
STAGE0_MODEL_PATH = STAGE0_OUTPUT_DIR / "best_stage0_feces_presence_resnet18.pt"
STAGE0_HISTORY_PATH = STAGE0_OUTPUT_DIR / "stage0_training_history.csv"
STAGE0_VAL_PREDS_PATH = STAGE0_OUTPUT_DIR / "stage0_val_predictions.csv"
STAGE0_TEST_PREDS_PATH = STAGE0_OUTPUT_DIR / "stage0_test_predictions.csv"

print("Stage 0 model path:", STAGE0_MODEL_PATH)

Stage 0 model path: C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\PII\chicken_feces_dataset\03_final\outputs_final_pipeline\stage0_feces_presence\best_stage0_feces_presence_resnet18.pt


# Stage 0 training

The Stage 0 gate is trained for a fixed number of epochs.

The best checkpoint is selected using validation macro F1.

In [39]:
STAGE0_NUM_EPOCHS = 12

stage0_history = []
best_stage0_val_macro_f1 = -1.0
best_stage0_epoch = None

for epoch in range(1, STAGE0_NUM_EPOCHS + 1):
    train_loss, train_acc, train_macro_f1 = train_one_epoch_stage0(
        model=stage0_model,
        loader=stage0_train_loader,
        criterion=stage0_criterion,
        optimizer=stage0_optimizer,
        device=device
    )

    _, val_metrics = evaluate_stage0(stage0_model, stage0_val_loader, device=device)

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "train_macro_f1": train_macro_f1,
        "val_acc": val_metrics["accuracy"],
        "val_macro_f1": val_metrics["macro_f1"]
    }
    stage0_history.append(row)

    if val_metrics["macro_f1"] > best_stage0_val_macro_f1:
        best_stage0_val_macro_f1 = val_metrics["macro_f1"]
        best_stage0_epoch = epoch
        torch.save(stage0_model.state_dict(), STAGE0_MODEL_PATH)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} | "
        f"train_acc={train_acc:.4f} | "
        f"train_macro_f1={train_macro_f1:.4f} | "
        f"val_acc={val_metrics['accuracy']:.4f} | "
        f"val_macro_f1={val_metrics['macro_f1']:.4f}"
    )

print("\nBest Stage 0 epoch:", best_stage0_epoch)
print("Best Stage 0 val_macro_f1:", round(best_stage0_val_macro_f1, 4))
print("Best checkpoint saved to:", STAGE0_MODEL_PATH)

Epoch 01 | train_loss=0.2712 | train_acc=0.8786 | train_macro_f1=0.8778 | val_acc=0.9667 | val_macro_f1=0.9666
Epoch 02 | train_loss=0.0709 | train_acc=0.9714 | train_macro_f1=0.9714 | val_acc=0.9667 | val_macro_f1=0.9666
Epoch 03 | train_loss=0.0079 | train_acc=1.0000 | train_macro_f1=1.0000 | val_acc=0.9667 | val_macro_f1=0.9666
Epoch 04 | train_loss=0.0077 | train_acc=1.0000 | train_macro_f1=1.0000 | val_acc=0.9667 | val_macro_f1=0.9666
Epoch 05 | train_loss=0.0068 | train_acc=1.0000 | train_macro_f1=1.0000 | val_acc=0.9667 | val_macro_f1=0.9666
Epoch 06 | train_loss=0.0044 | train_acc=1.0000 | train_macro_f1=1.0000 | val_acc=0.9667 | val_macro_f1=0.9666
Epoch 07 | train_loss=0.0035 | train_acc=1.0000 | train_macro_f1=1.0000 | val_acc=1.0000 | val_macro_f1=1.0000
Epoch 08 | train_loss=0.0041 | train_acc=1.0000 | train_macro_f1=1.0000 | val_acc=0.9667 | val_macro_f1=0.9666
Epoch 09 | train_loss=0.0109 | train_acc=0.9929 | train_macro_f1=0.9929 | val_acc=0.9667 | val_macro_f1=0.9666
E

# Stage 0 training history

The epoch-level training history is stored as a dataframe for inspection and export.

In [40]:
stage0_history_df = pd.DataFrame(stage0_history)
stage0_history_df.to_csv(STAGE0_HISTORY_PATH, index=False)

display(stage0_history_df)
print("Training history saved to:", STAGE0_HISTORY_PATH)

,epoch,train_loss,train_acc,train_macro_f1,val_acc,val_macro_f1
0,1,0.271155,0.878571,0.877817,0.966667,0.96663
1,2,0.070929,0.971429,0.971405,0.966667,0.96663
2,3,0.007922,1.000000,1.000000,0.966667,0.96663
3,4,0.007739,1.000000,1.000000,0.966667,0.96663
4,5,0.006800,1.000000,1.000000,0.966667,0.96663
5,6,0.004397,1.000000,1.000000,0.966667,0.96663
6,7,0.003475,1.000000,1.000000,1.000000,1.00000
7,8,0.004128,1.000000,1.000000,0.966667,0.96663
8,9,0.010878,0.992857,0.992857,0.966667,0.96663
9,10,0.000986,1.000000,1.000000,0.966667,0.96663


Training history saved to: C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\PII\chicken_feces_dataset\03_final\outputs_final_pipeline\stage0_feces_presence\stage0_training_history.csv


# Stage 0 best-checkpoint loading

The best validation checkpoint is reloaded before final validation and test evaluation.

In [41]:
best_stage0_model = models.resnet18(weights=None)
best_stage0_model.fc = nn.Linear(best_stage0_model.fc.in_features, 2)
best_stage0_model.load_state_dict(torch.load(STAGE0_MODEL_PATH, map_location=device))
best_stage0_model = best_stage0_model.to(device)
best_stage0_model.eval()

print("Best Stage 0 model loaded successfully.")

Best Stage 0 model loaded successfully.


# Stage 0 validation evaluation

The best checkpoint is evaluated on the validation split.

In [42]:
stage0_val_results, stage0_val_metrics = evaluate_stage0(
    best_stage0_model,
    stage0_val_loader,
    device=device
)

stage0_val_results.to_csv(STAGE0_VAL_PREDS_PATH, index=False)

display(stage0_val_results.head())
print("Stage 0 validation accuracy:", round(stage0_val_metrics["accuracy"], 4))
print("Stage 0 validation macro F1:", round(stage0_val_metrics["macro_f1"], 4))
print("\nStage 0 validation confusion matrix:")
print(stage0_val_metrics["confusion_matrix"])
print("\nStage 0 validation classification report:")
print(stage0_val_metrics["classification_report"])

,filename,filepath,true_label,pred_label,correct,confidence,prob_feces_present,prob_no_feces
0,feces_present_002.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,feces_present,feces_present,True,0.999980,0.999980,0.000020
1,no_feces_096.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,no_feces,no_feces,True,0.996810,0.003190,0.996810
2,feces_present_064.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,feces_present,feces_present,True,0.999922,0.999922,0.000078
3,feces_present_075.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,feces_present,feces_present,True,0.999369,0.999369,0.000631
4,feces_present_053.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,feces_present,feces_present,True,0.999973,0.999973,0.000027


Stage 0 validation accuracy: 1.0
Stage 0 validation macro F1: 1.0

Stage 0 validation confusion matrix:
[[15  0]
 [ 0 15]]

Stage 0 validation classification report:
               precision    recall  f1-score   support

feces_present       1.00      1.00      1.00        15
     no_feces       1.00      1.00      1.00        15

     accuracy                           1.00        30
    macro avg       1.00      1.00      1.00        30
 weighted avg       1.00      1.00      1.00        30



# Stage 0 test evaluation

The best checkpoint is evaluated on the held-out test split.

In [43]:
stage0_test_results, stage0_test_metrics = evaluate_stage0(
    best_stage0_model,
    stage0_test_loader,
    device=device
)

stage0_test_results.to_csv(STAGE0_TEST_PREDS_PATH, index=False)

display(stage0_test_results.head())
print("Stage 0 test accuracy:", round(stage0_test_metrics["accuracy"], 4))
print("Stage 0 test macro F1:", round(stage0_test_metrics["macro_f1"], 4))
print("\nStage 0 test confusion matrix:")
print(stage0_test_metrics["confusion_matrix"])
print("\nStage 0 test classification report:")
print(stage0_test_metrics["classification_report"])

print("\nValidation predictions saved to:", STAGE0_VAL_PREDS_PATH)
print("Test predictions saved to:", STAGE0_TEST_PREDS_PATH)

,filename,filepath,true_label,pred_label,correct,confidence,prob_feces_present,prob_no_feces
0,feces_present_042.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,feces_present,feces_present,True,0.999683,0.999683,3.167100e-04
1,feces_present_049.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,feces_present,feces_present,True,0.999999,0.999999,9.392194e-07
2,feces_present_099.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,feces_present,feces_present,True,0.990906,0.990906,9.094069e-03
3,feces_present_095.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,feces_present,feces_present,True,0.999616,0.999616,3.842739e-04
4,no_feces_041.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,no_feces,no_feces,True,0.999775,0.000225,9.997751e-01


Stage 0 test accuracy: 1.0
Stage 0 test macro F1: 1.0

Stage 0 test confusion matrix:
[[15  0]
 [ 0 15]]

Stage 0 test classification report:
               precision    recall  f1-score   support

feces_present       1.00      1.00      1.00        15
     no_feces       1.00      1.00      1.00        15

     accuracy                           1.00        30
    macro avg       1.00      1.00      1.00        30
 weighted avg       1.00      1.00      1.00        30


Validation predictions saved to: C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\PII\chicken_feces_dataset\03_final\outputs_final_pipeline\stage0_feces_presence\stage0_val_predictions.csv
Test predictions saved to: C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\PII\chicken_feces_dataset\03_final\outputs_final_pipeline\stage0_feces_presence\stage0_test_predictions.csv


# Stage 0 interpretation checkpoint

At this point, the feces-presence gate has been trained and evaluated on validation and test data.

Before proceeding to the next stage of the hierarchical pipeline, the Stage 0 results should be reviewed to assess whether the auxiliary gate is learning a meaningful and stable decision boundary.

# Stage 0 results and interpretation

The Stage 0 feces-presence gate achieved excellent internal performance on the auxiliary dataset.

## Summary of results

- the best checkpoint was selected at **epoch 7** based on validation macro F1,
- validation performance reached:
  - **accuracy = 1.0000**
  - **macro F1 = 1.0000**
- test performance reached:
  - **accuracy = 1.0000**
  - **macro F1 = 1.0000**

Both the validation and test confusion matrices showed perfect separation between:

- `feces_present`
- `no_feces`

## Interpretation

These results indicate that, within the auxiliary Stage 0 dataset, the binary distinction between visible fecal samples and non-fecal floor/background images is highly learnable.

The training dynamics were stable, and the validation and test behavior were consistent, which supports the use of this module as the first stage of the final hierarchical pipeline.

## Caution

Despite the excellent internal performance, this result should be interpreted carefully.

The Stage 0 dataset is auxiliary and externally assembled, which means that perfect internal performance does not automatically guarantee robust behavior when the gate is applied to the main feces dataset or to external real-world inputs.

For this reason, the next step is to evaluate the Stage 0 gate on the large feces dataset, where all images should ideally be accepted as `feces_present`. This screening step is necessary to quantify the false rejection rate on valid fecal samples before integrating the gate into the full pipeline.

# Stage 0 screening on the large feces dataset

After training and internally evaluating the Stage 0 feces-presence gate on the auxiliary dataset, the next step is to test its behavior on the large feces dataset used in the main project.

## Purpose of this screening step

All images in the large clinical dataset are expected to contain visible fecal samples. Therefore, when Stage 0 is applied to this dataset, the desired behavior is:

- accept all samples as `feces_present`,
- reject as few valid fecal images as possible.

## Main question

This screening step is designed to estimate the **false rejection rate on valid feces images**, which is critical before integrating Stage 0 into the full hierarchical pipeline.

A gate that performs well on its own auxiliary dataset but incorrectly rejects many valid fecal samples would interfere with the downstream health-status and disease classifiers.

# Large-dataset loading for Stage 0 screening

For the screening analysis, the large feces dataset is reconstructed from the rebuilt multiclass splits:

- `train.csv`
- `val.csv`
- `test.csv`

These files are merged into a single dataframe because the goal here is not to retrain Stage 0, but to examine how the trained gate behaves on the full set of valid fecal images.

In [44]:
MULTICLASS_TRAIN_CSV = MULTICLASS_SPLITS_DIR / "train.csv"
MULTICLASS_VAL_CSV = MULTICLASS_SPLITS_DIR / "val.csv"
MULTICLASS_TEST_CSV = MULTICLASS_SPLITS_DIR / "test.csv"

large_train_df = pd.read_csv(MULTICLASS_TRAIN_CSV).copy()
large_val_df = pd.read_csv(MULTICLASS_VAL_CSV).copy()
large_test_df = pd.read_csv(MULTICLASS_TEST_CSV).copy()

large_feces_df = pd.concat(
    [large_train_df, large_val_df, large_test_df],
    axis=0,
    ignore_index=True
)

print("Large feces dataset size:", len(large_feces_df))
display(large_feces_df.head())

Large feces dataset size: 9621


,image_id,filename,image_path,label
0,CFD_06022,CFD_06022.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,newcastle
1,CFD_01975,CFD_01975.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,coccidiosis
2,CFD_05643,CFD_05643.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,healthy
3,CFD_06896,CFD_06896.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,salmonella
4,CFD_06784,CFD_06784.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,salmonella


# Path reconstruction for the large feces dataset

In accordance with the project's technical rule, image paths are reconstructed using `filename` rather than `image_path`.

In [45]:
FINAL_IMAGES_DIR = ROOT / "03_final" / "images"

def build_large_feces_path(filename: str) -> Path:
    return FINAL_IMAGES_DIR / filename

large_feces_df["filepath"] = large_feces_df["filename"].apply(build_large_feces_path)
large_feces_df["exists"] = large_feces_df["filepath"].apply(Path.exists)

print(large_feces_df["exists"].value_counts())

if not large_feces_df["exists"].all():
    missing_large_files = large_feces_df.loc[~large_feces_df["exists"], ["filename", "filepath"]]
    display(missing_large_files.head(20))
    raise FileNotFoundError("Some files in the large feces dataset were not found.")
else:
    print("All large-dataset files were found successfully.")

exists
True    9621
Name: count, dtype: int64
All large-dataset files were found successfully.


# Screening dataset definition

A lightweight dataset is defined for inference-only screening with the trained Stage 0 model.

Since all samples in this screening set are known to contain feces, no diagnostic labels are needed for this step. The goal is only to inspect the Stage 0 predictions.

In [46]:
class Stage0ScreeningDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None):
        self.df = df.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = Path(row["filepath"])

        try:
            image = Image.open(image_path).convert("RGB")
        except Exception as e:
            raise RuntimeError(f"Error opening screening image: {image_path} | {e}")

        if self.transform is not None:
            image = self.transform(image)

        return image, row["filename"], str(image_path)

# Screening dataloader

The large feces dataset is wrapped into a dataloader for batch inference with the trained Stage 0 gate.

In [47]:
stage0_screening_dataset = Stage0ScreeningDataset(
    large_feces_df,
    transform=stage0_eval_transform
)

stage0_screening_loader = DataLoader(
    stage0_screening_dataset,
    batch_size=32,
    shuffle=False
)

print("Screening batches:", len(stage0_screening_loader))

Screening batches: 301


# Stage 0 screening inference

The trained Stage 0 checkpoint is now applied to the large feces dataset.

Because all samples in this dataset are valid fecal images, the expected prediction for every image is `feces_present`.

In [48]:
@torch.no_grad()
def screen_large_feces_with_stage0(model, loader, device):
    model.eval()
    rows = []

    for images, filenames, filepaths in loader:
        images = images.to(device)

        logits = model(images)
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        for i in range(len(filenames)):
            pred_idx = int(preds[i].cpu().item())
            pred_label = stage0_idx_to_class[pred_idx]

            rows.append({
                "filename": filenames[i],
                "filepath": filepaths[i],
                "stage0_pred_label": pred_label,
                "stage0_confidence": float(probs[i].max().cpu().item()),
                "prob_feces_present": float(probs[i][stage0_class_to_idx["feces_present"]].cpu().item()),
                "prob_no_feces": float(probs[i][stage0_class_to_idx["no_feces"]].cpu().item())
            })

    return pd.DataFrame(rows)

In [49]:
stage0_screening_results = screen_large_feces_with_stage0(
    best_stage0_model,
    stage0_screening_loader,
    device=device
)

display(stage0_screening_results.head())
print("Total screened samples:", len(stage0_screening_results))

,filename,filepath,stage0_pred_label,stage0_confidence,prob_feces_present,prob_no_feces
0,CFD_06022.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,feces_present,0.994772,0.994772,0.005228
1,CFD_01975.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,feces_present,0.997998,0.997998,0.002002
2,CFD_05643.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,feces_present,0.989814,0.989814,0.010186
3,CFD_06896.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,feces_present,0.971488,0.971488,0.028512
4,CFD_06784.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,feces_present,0.563007,0.563007,0.436993


Total screened samples: 9621


# Screening summary metrics

Since all images in the large dataset are expected to contain feces, the main quantity of interest is the proportion of samples incorrectly rejected as `no_feces`.

In [50]:
num_total = len(stage0_screening_results)
num_accepted = (stage0_screening_results["stage0_pred_label"] == "feces_present").sum()
num_rejected = (stage0_screening_results["stage0_pred_label"] == "no_feces").sum()

acceptance_rate = num_accepted / num_total
false_rejection_rate = num_rejected / num_total

print("Stage 0 acceptance rate on valid feces images:", round(acceptance_rate, 4))
print("Stage 0 false rejection rate on valid feces images:", round(false_rejection_rate, 4))

Stage 0 acceptance rate on valid feces images: 0.893
Stage 0 false rejection rate on valid feces images: 0.107


# Rejected-sample inspection

Any image predicted as `no_feces` at this stage represents a false rejection on a valid fecal sample.

These cases should be collected for later inspection because they reveal whether the gate is likely to interfere with the downstream stages of the pipeline.

In [51]:
stage0_false_rejections = stage0_screening_results[
    stage0_screening_results["stage0_pred_label"] == "no_feces"
].copy()

print("False rejections:", len(stage0_false_rejections))
display(stage0_false_rejections.head(20))

False rejections: 1029


,filename,filepath,stage0_pred_label,stage0_confidence,prob_feces_present,prob_no_feces
6,CFD_05093.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,no_feces,0.905172,0.094828,0.905172
9,CFD_05206.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,no_feces,0.920366,0.079634,0.920366
23,CFD_06469.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,no_feces,0.672782,0.327218,0.672782
29,CFD_03908.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,no_feces,0.577497,0.422503,0.577497
36,CFD_06099.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,no_feces,0.523054,0.476946,0.523054
45,CFD_05319.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,no_feces,0.905782,0.094218,0.905782
55,CFD_05486.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,no_feces,0.726265,0.273735,0.726265
68,CFD_04306.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,no_feces,0.780010,0.219990,0.780010
80,CFD_09238.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,no_feces,0.732928,0.267072,0.732928
81,CFD_04864.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,no_feces,0.947930,0.052070,0.947930


# Screening results export

The full screening predictions and the subset of false rejections are saved for later analysis.

In [52]:
STAGE0_SCREENING_PREDS_PATH = STAGE0_OUTPUT_DIR / "stage0_screening_large_feces_predictions.csv"
STAGE0_FALSE_REJECTIONS_PATH = STAGE0_OUTPUT_DIR / "stage0_screening_large_feces_false_rejections.csv"

stage0_screening_results.to_csv(STAGE0_SCREENING_PREDS_PATH, index=False)
stage0_false_rejections.to_csv(STAGE0_FALSE_REJECTIONS_PATH, index=False)

print("Screening predictions saved to:", STAGE0_SCREENING_PREDS_PATH)
print("False rejections saved to:", STAGE0_FALSE_REJECTIONS_PATH)

Screening predictions saved to: C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\PII\chicken_feces_dataset\03_final\outputs_final_pipeline\stage0_feces_presence\stage0_screening_large_feces_predictions.csv
False rejections saved to: C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\PII\chicken_feces_dataset\03_final\outputs_final_pipeline\stage0_feces_presence\stage0_screening_large_feces_false_rejections.csv


# Stage 0 screening interpretation checkpoint

The large-dataset screening results should now be reviewed before Stage 0 is considered ready for integration into the final hierarchical pipeline.

The key question is whether the feces-presence gate preserves a high acceptance rate on valid fecal samples while keeping the false rejection rate low.

# Stage 0 screening results on the large feces dataset

After internal evaluation on the auxiliary Stage 0 dataset, the feces-presence gate was applied to the large feces dataset used in the main project.

## Screening objective

Because all images in the large dataset are known to contain visible fecal samples, the expected behavior of Stage 0 is to classify all of them as `feces_present`.

## Observed behavior

The screening results showed:

- **acceptance rate on valid fecal images = 0.8930**
- **false rejection rate on valid fecal images = 0.1070**
- **1029 valid fecal images** were incorrectly classified as `no_feces`

## Interpretation

These results show that the excellent internal performance obtained on the auxiliary Stage 0 dataset does not fully transfer to the large feces dataset.

In other words, the binary distinction between `feces_present` and `no_feces` is highly learnable within the auxiliary dataset, but the resulting decision boundary is not yet robust enough to be used as a strict front-end rejection gate in the final hierarchical pipeline.

A false rejection rate of 10.7% is too high for this module to block samples before they reach the downstream health-status and disease classifiers.

## Methodological conclusion

At the current stage, Stage 0 should be interpreted as an exploratory auxiliary module rather than as a fully validated first-stage filter.

Its internal results are promising, but its behavior on the large feces dataset indicates a relevant domain-transfer limitation. Therefore, it should not yet be used as a hard rejection gate in the final pipeline without further refinement.

# Stage 1 — Health-status classifier

The second stage of the hierarchical pipeline is a binary health-status classifier applied only to images that are considered valid fecal samples.

## Stage definition

The two classes for this stage are:

- `healthy`
- `unhealthy`

## Role within the full pipeline

This stage is responsible for separating normal fecal samples from pathological or suspicious ones.

Its output determines whether the sample:

- exits the pipeline as `healthy`, or
- proceeds to the disease-specific classifier in Stage 2.

## Methodological note

This stage corresponds to the main healthy-versus-pathological decision boundary of the project and is therefore a critical component of the final pipeline.

# Stage 1 dataset loading

This section loads the rebuilt hierarchical binary splits:

- `train_binary.csv`
- `val_binary.csv`
- `test_binary.csv`

These splits are used as the official data source for the Stage 1 health-status classifier.

In [53]:
BINARY_TRAIN_CSV = HIERARCHICAL_SPLITS_DIR / "train_binary.csv"
BINARY_VAL_CSV = HIERARCHICAL_SPLITS_DIR / "val_binary.csv"
BINARY_TEST_CSV = HIERARCHICAL_SPLITS_DIR / "test_binary.csv"

stage1_train_df = pd.read_csv(BINARY_TRAIN_CSV).copy()
stage1_val_df = pd.read_csv(BINARY_VAL_CSV).copy()
stage1_test_df = pd.read_csv(BINARY_TEST_CSV).copy()

print("Stage 1 train size:", len(stage1_train_df))
print(stage1_train_df["label"].value_counts())
print()

print("Stage 1 val size:", len(stage1_val_df))
print(stage1_val_df["label"].value_counts())
print()

print("Stage 1 test size:", len(stage1_test_df))
print(stage1_test_df["label"].value_counts())

Stage 1 train size: 6734
label
unhealthy    4638
healthy      2096
Name: count, dtype: int64

Stage 1 val size: 1443
label
unhealthy    994
healthy      449
Name: count, dtype: int64

Stage 1 test size: 1444
label
unhealthy    994
healthy      450
Name: count, dtype: int64


# Stage 1 path reconstruction

As in the rest of the project, image paths are reconstructed using `filename` rather than `image_path`.

In [54]:
def build_stage1_path(filename: str) -> Path:
    return FINAL_IMAGES_DIR / filename

for df in [stage1_train_df, stage1_val_df, stage1_test_df]:
    df["filepath"] = df["filename"].apply(build_stage1_path)
    df["exists"] = df["filepath"].apply(Path.exists)

print("Stage 1 train exists:")
print(stage1_train_df["exists"].value_counts())
print()

print("Stage 1 val exists:")
print(stage1_val_df["exists"].value_counts())
print()

print("Stage 1 test exists:")
print(stage1_test_df["exists"].value_counts())

if not stage1_train_df["exists"].all():
    raise FileNotFoundError("Some Stage 1 train images were not found.")
if not stage1_val_df["exists"].all():
    raise FileNotFoundError("Some Stage 1 validation images were not found.")
if not stage1_test_df["exists"].all():
    raise FileNotFoundError("Some Stage 1 test images were not found.")

Stage 1 train exists:
exists
True    6734
Name: count, dtype: int64

Stage 1 val exists:
exists
True    1443
Name: count, dtype: int64

Stage 1 test exists:
exists
True    1444
Name: count, dtype: int64


# Stage 1 label encoding

The binary label encoding for the health-status classifier is defined as follows:

- `healthy` -> `0`
- `unhealthy` -> `1`

In [55]:
stage1_class_to_idx = {
    "healthy": 0,
    "unhealthy": 1
}

stage1_idx_to_class = {v: k for k, v in stage1_class_to_idx.items()}

for df in [stage1_train_df, stage1_val_df, stage1_test_df]:
    df["label_idx"] = df["label"].map(stage1_class_to_idx)

print("Stage 1 label encoding completed.")
print(stage1_train_df[["filename", "label", "label_idx"]].head())

Stage 1 label encoding completed.
        filename      label  label_idx
0  CFD_06022.jpg  unhealthy          1
1  CFD_01975.jpg  unhealthy          1
2  CFD_05643.jpg    healthy          0
3  CFD_06896.jpg  unhealthy          1
4  CFD_06784.jpg  unhealthy          1


# Stage 1 preprocessing

The health-status classifier uses the same image resolution as the rest of the pipeline.

Training uses lightweight augmentation, while validation and test use deterministic preprocessing.

In [56]:
stage1_train_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor()
])

stage1_eval_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor()
])

# Stage 1 dataset class

A dedicated PyTorch dataset is defined for the health-status classification stage.

Each sample returns:

- image tensor
- numeric label
- filename
- resolved filepath

In [57]:
class Stage1Dataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None):
        self.df = df.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = Path(row["filepath"])

        try:
            image = Image.open(image_path).convert("RGB")
        except Exception as e:
            raise RuntimeError(f"Error opening Stage 1 image: {image_path} | {e}")

        label = int(row["label_idx"])

        if self.transform is not None:
            image = self.transform(image)

        return image, label, row["filename"], str(image_path)

# Stage 1 datasets and dataloaders

The train, validation, and test partitions are instantiated as PyTorch datasets and wrapped into dataloaders.

In [58]:
stage1_train_dataset = Stage1Dataset(stage1_train_df, transform=stage1_train_transform)
stage1_val_dataset = Stage1Dataset(stage1_val_df, transform=stage1_eval_transform)
stage1_test_dataset = Stage1Dataset(stage1_test_df, transform=stage1_eval_transform)

stage1_train_loader = DataLoader(stage1_train_dataset, batch_size=32, shuffle=True)
stage1_val_loader = DataLoader(stage1_val_dataset, batch_size=32, shuffle=False)
stage1_test_loader = DataLoader(stage1_test_dataset, batch_size=32, shuffle=False)

print("Stage 1 loaders created successfully.")
print("Train batches:", len(stage1_train_loader))
print("Val batches:  ", len(stage1_val_loader))
print("Test batches: ", len(stage1_test_loader))

Stage 1 loaders created successfully.
Train batches: 211
Val batches:   46
Test batches:  46


# Stage 1 model definition

The health-status classifier is implemented with a pretrained ResNet-18 backbone.

The final fully connected layer is replaced with a two-output linear classifier corresponding to the two Stage 1 classes.

In [59]:
stage1_model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
stage1_in_features = stage1_model.fc.in_features
stage1_model.fc = nn.Linear(stage1_in_features, 2)
stage1_model = stage1_model.to(device)

stage1_criterion = nn.CrossEntropyLoss()
stage1_optimizer = torch.optim.Adam(stage1_model.parameters(), lr=1e-4)

print(stage1_model.fc)

Linear(in_features=512, out_features=2, bias=True)


# Stage 1 training and evaluation utilities

This section defines helper functions for:

- one training epoch,
- full-dataset prediction,
- metric computation,
- checkpoint selection based on validation macro F1.

In [60]:
def train_one_epoch_stage1(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    y_true = []
    y_pred = []

    for images, labels, _, _ in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(logits, dim=1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(y_true, y_pred)
    epoch_macro_f1 = f1_score(y_true, y_pred, average="macro")

    return epoch_loss, epoch_acc, epoch_macro_f1


@torch.no_grad()
def predict_stage1(model, loader, device):
    model.eval()
    rows = []

    for images, labels, filenames, filepaths in loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        for i in range(len(filenames)):
            true_idx = int(labels[i].cpu().item())
            pred_idx = int(preds[i].cpu().item())

            rows.append({
                "filename": filenames[i],
                "filepath": filepaths[i],
                "true_label": stage1_idx_to_class[true_idx],
                "pred_label": stage1_idx_to_class[pred_idx],
                "correct": true_idx == pred_idx,
                "confidence": float(probs[i].max().cpu().item()),
                "prob_healthy": float(probs[i][stage1_class_to_idx["healthy"]].cpu().item()),
                "prob_unhealthy": float(probs[i][stage1_class_to_idx["unhealthy"]].cpu().item())
            })

    return pd.DataFrame(rows)


def evaluate_stage1(model, loader, device):
    results_df = predict_stage1(model, loader, device=device)

    y_true = results_df["true_label"]
    y_pred = results_df["pred_label"]

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=["healthy", "unhealthy"]),
        "classification_report": classification_report(
            y_true,
            y_pred,
            labels=["healthy", "unhealthy"],
            zero_division=0
        )
    }

    return results_df, metrics

# Stage 1 output paths

Dedicated output paths are defined for:

- best checkpoint,
- training history,
- validation predictions,
- test predictions.

In [61]:
STAGE1_MODEL_PATH = STAGE1_OUTPUT_DIR / "best_stage1_binary_health_resnet18.pt"
STAGE1_HISTORY_PATH = STAGE1_OUTPUT_DIR / "stage1_training_history.csv"
STAGE1_VAL_PREDS_PATH = STAGE1_OUTPUT_DIR / "stage1_val_predictions.csv"
STAGE1_TEST_PREDS_PATH = STAGE1_OUTPUT_DIR / "stage1_test_predictions.csv"

print("Stage 1 model path:", STAGE1_MODEL_PATH)

Stage 1 model path: C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\PII\chicken_feces_dataset\03_final\outputs_final_pipeline\stage1_binary_health\best_stage1_binary_health_resnet18.pt


# Stage 1 training

The Stage 1 health-status classifier is trained for a fixed number of epochs.

The best checkpoint is selected using validation macro F1.

In [62]:
STAGE1_NUM_EPOCHS = 12

stage1_history = []
best_stage1_val_macro_f1 = -1.0
best_stage1_epoch = None

for epoch in range(1, STAGE1_NUM_EPOCHS + 1):
    train_loss, train_acc, train_macro_f1 = train_one_epoch_stage1(
        model=stage1_model,
        loader=stage1_train_loader,
        criterion=stage1_criterion,
        optimizer=stage1_optimizer,
        device=device
    )

    _, val_metrics = evaluate_stage1(stage1_model, stage1_val_loader, device=device)

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "train_macro_f1": train_macro_f1,
        "val_acc": val_metrics["accuracy"],
        "val_macro_f1": val_metrics["macro_f1"]
    }
    stage1_history.append(row)

    if val_metrics["macro_f1"] > best_stage1_val_macro_f1:
        best_stage1_val_macro_f1 = val_metrics["macro_f1"]
        best_stage1_epoch = epoch
        torch.save(stage1_model.state_dict(), STAGE1_MODEL_PATH)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} | "
        f"train_acc={train_acc:.4f} | "
        f"train_macro_f1={train_macro_f1:.4f} | "
        f"val_acc={val_metrics['accuracy']:.4f} | "
        f"val_macro_f1={val_metrics['macro_f1']:.4f}"
    )

print("\nBest Stage 1 epoch:", best_stage1_epoch)
print("Best Stage 1 val_macro_f1:", round(best_stage1_val_macro_f1, 4))
print("Best checkpoint saved to:", STAGE1_MODEL_PATH)

Epoch 01 | train_loss=0.1579 | train_acc=0.9375 | train_macro_f1=0.9272 | val_acc=0.9432 | val_macro_f1=0.9305
Epoch 02 | train_loss=0.0777 | train_acc=0.9740 | train_macro_f1=0.9696 | val_acc=0.9633 | val_macro_f1=0.9568
Epoch 03 | train_loss=0.0523 | train_acc=0.9813 | train_macro_f1=0.9782 | val_acc=0.9771 | val_macro_f1=0.9733
Epoch 04 | train_loss=0.0377 | train_acc=0.9854 | train_macro_f1=0.9830 | val_acc=0.9709 | val_macro_f1=0.9656
Epoch 05 | train_loss=0.0328 | train_acc=0.9874 | train_macro_f1=0.9853 | val_acc=0.9820 | val_macro_f1=0.9791
Epoch 06 | train_loss=0.0227 | train_acc=0.9944 | train_macro_f1=0.9934 | val_acc=0.9792 | val_macro_f1=0.9755
Epoch 07 | train_loss=0.0239 | train_acc=0.9918 | train_macro_f1=0.9905 | val_acc=0.9778 | val_macro_f1=0.9740
Epoch 08 | train_loss=0.0183 | train_acc=0.9935 | train_macro_f1=0.9924 | val_acc=0.9792 | val_macro_f1=0.9756
Epoch 09 | train_loss=0.0192 | train_acc=0.9936 | train_macro_f1=0.9926 | val_acc=0.9716 | val_macro_f1=0.9662
E

# Stage 1 training history

The epoch-level training history is stored as a dataframe for inspection and export.

In [63]:
stage1_history_df = pd.DataFrame(stage1_history)
stage1_history_df.to_csv(STAGE1_HISTORY_PATH, index=False)

display(stage1_history_df)
print("Training history saved to:", STAGE1_HISTORY_PATH)

,epoch,train_loss,train_acc,train_macro_f1,val_acc,val_macro_f1
0,1,0.157885,0.937481,0.927178,0.943174,0.930460
1,2,0.077688,0.974012,0.969610,0.963271,0.956813
2,3,0.052344,0.981289,0.978151,0.977131,0.973277
3,4,0.037668,0.985447,0.983011,0.970894,0.965580
4,5,0.032766,0.987377,0.985290,0.981982,0.979060
5,6,0.022659,0.994357,0.993419,0.979210,0.975539
6,7,0.023861,0.991832,0.990474,0.977824,0.974006
7,8,0.018284,0.993466,0.992380,0.979210,0.975601
8,9,0.019150,0.993614,0.992558,0.971587,0.966157
9,10,0.010863,0.996881,0.996365,0.968122,0.963216


Training history saved to: C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\PII\chicken_feces_dataset\03_final\outputs_final_pipeline\stage1_binary_health\stage1_training_history.csv


# Stage 1 best-checkpoint loading

The best validation checkpoint is reloaded before final validation and test evaluation.

In [64]:
best_stage1_model = models.resnet18(weights=None)
best_stage1_model.fc = nn.Linear(best_stage1_model.fc.in_features, 2)
best_stage1_model.load_state_dict(torch.load(STAGE1_MODEL_PATH, map_location=device))
best_stage1_model = best_stage1_model.to(device)
best_stage1_model.eval()

print("Best Stage 1 model loaded successfully.")

Best Stage 1 model loaded successfully.


# Stage 1 validation evaluation

The best checkpoint is evaluated on the validation split.

In [65]:
stage1_val_results, stage1_val_metrics = evaluate_stage1(
    best_stage1_model,
    stage1_val_loader,
    device=device
)

stage1_val_results.to_csv(STAGE1_VAL_PREDS_PATH, index=False)

display(stage1_val_results.head())
print("Stage 1 validation accuracy:", round(stage1_val_metrics["accuracy"], 4))
print("Stage 1 validation macro F1:", round(stage1_val_metrics["macro_f1"], 4))
print("\nStage 1 validation confusion matrix:")
print(stage1_val_metrics["confusion_matrix"])
print("\nStage 1 validation classification report:")
print(stage1_val_metrics["classification_report"])

,filename,filepath,true_label,pred_label,correct,confidence,prob_healthy,prob_unhealthy
0,CFD_02458.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,unhealthy,unhealthy,True,0.999902,0.000097,0.999902
1,CFD_04303.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,healthy,healthy,True,0.986170,0.986170,0.013830
2,CFD_06694.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,unhealthy,unhealthy,True,0.956957,0.043043,0.956957
3,CFD_09550.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,unhealthy,unhealthy,True,0.999965,0.000035,0.999965
4,CFD_06501.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,unhealthy,unhealthy,True,0.999166,0.000834,0.999166


Stage 1 validation accuracy: 0.982
Stage 1 validation macro F1: 0.9791

Stage 1 validation confusion matrix:
[[439  10]
 [ 16 978]]

Stage 1 validation classification report:
              precision    recall  f1-score   support

     healthy       0.96      0.98      0.97       449
   unhealthy       0.99      0.98      0.99       994

    accuracy                           0.98      1443
   macro avg       0.98      0.98      0.98      1443
weighted avg       0.98      0.98      0.98      1443



# Stage 1 test evaluation

The best checkpoint is evaluated on the held-out test split.

In [66]:
stage1_test_results, stage1_test_metrics = evaluate_stage1(
    best_stage1_model,
    stage1_test_loader,
    device=device
)

stage1_test_results.to_csv(STAGE1_TEST_PREDS_PATH, index=False)

display(stage1_test_results.head())
print("Stage 1 test accuracy:", round(stage1_test_metrics["accuracy"], 4))
print("Stage 1 test macro F1:", round(stage1_test_metrics["macro_f1"], 4))
print("\nStage 1 test confusion matrix:")
print(stage1_test_metrics["confusion_matrix"])
print("\nStage 1 test classification report:")
print(stage1_test_metrics["classification_report"])

print("\nValidation predictions saved to:", STAGE1_VAL_PREDS_PATH)
print("Test predictions saved to:", STAGE1_TEST_PREDS_PATH)

,filename,filepath,true_label,pred_label,correct,confidence,prob_healthy,prob_unhealthy
0,CFD_04544.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,healthy,healthy,True,1.000000,9.999995e-01,4.318456e-07
1,CFD_08622.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,unhealthy,unhealthy,True,1.000000,3.347282e-08,1.000000e+00
2,CFD_01667.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,unhealthy,unhealthy,True,0.999926,7.392649e-05,9.999261e-01
3,CFD_08158.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,unhealthy,unhealthy,True,0.999995,5.387720e-06,9.999946e-01
4,CFD_07551.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,unhealthy,unhealthy,True,0.999984,1.569718e-05,9.999843e-01


Stage 1 test accuracy: 0.9785
Stage 1 test macro F1: 0.9751

Stage 1 test confusion matrix:
[[437  13]
 [ 18 976]]

Stage 1 test classification report:
              precision    recall  f1-score   support

     healthy       0.96      0.97      0.97       450
   unhealthy       0.99      0.98      0.98       994

    accuracy                           0.98      1444
   macro avg       0.97      0.98      0.98      1444
weighted avg       0.98      0.98      0.98      1444


Validation predictions saved to: C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\PII\chicken_feces_dataset\03_final\outputs_final_pipeline\stage1_binary_health\stage1_val_predictions.csv
Test predictions saved to: C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\PII\chicken_feces_dataset\03_final\outputs_final_pipeline\stage1_binary_health\stage1_test_predictions.csv


# Stage 1 interpretation checkpoint

At this point, the health-status classifier has been trained and evaluated on validation and test data.

Before proceeding to Stage 2, the Stage 1 results should be reviewed in order to assess whether the healthy-versus-unhealthy decision boundary is sufficiently strong and stable within the rebuilt hierarchical binary setting.

# Stage 1 results and interpretation

The Stage 1 health-status classifier showed strong and stable internal performance on the rebuilt binary hierarchical splits.

## Summary of results

The best checkpoint was selected at **epoch 5** based on validation macro F1.

### Validation performance
- **accuracy = 0.9820**
- **macro F1 = 0.9791**

### Test performance
- **accuracy = 0.9785**
- **macro F1 = 0.9751**

## Confusion-matrix interpretation

### Validation confusion matrix
- `healthy`: 439 correctly classified, 10 misclassified as `unhealthy`
- `unhealthy`: 978 correctly classified, 16 misclassified as `healthy`

### Test confusion matrix
- `healthy`: 437 correctly classified, 13 misclassified as `unhealthy`
- `unhealthy`: 976 correctly classified, 18 misclassified as `healthy`

These results indicate that the binary health-status classifier is internally robust and generalizes consistently from validation to test within the rebuilt dataset setting.

## Interpretation

The Stage 1 classifier provides a strong internal healthy-versus-unhealthy decision boundary, with high accuracy and high macro F1 on both validation and test data.

The error pattern is limited and relatively balanced, although `healthy` remains slightly more delicate than `unhealthy`, which is consistent with the broader project diagnosis.

## Methodological conclusion

At the internal evaluation level, Stage 1 can be considered a reliable component of the hierarchical pipeline.

However, these results should still be interpreted in conjunction with the project's broader findings: strong internal binary performance does not automatically guarantee equally strong external generalization, particularly on difficult healthy variants and borderline cases.

# Stage 2 — Disease-specific classifier

The third stage of the hierarchical pipeline is a disease-specific multiclass classifier applied only to fecal samples that have already been classified as `unhealthy`.

## Stage definition

The three classes for this stage are:

- `coccidiosis`
- `newcastle`
- `salmonella`

## Role within the full pipeline

This stage is responsible for assigning a specific disease label to samples that were previously identified as pathological by Stage 1.

## Methodological note

This stage does not attempt to separate healthy from unhealthy samples. Instead, it focuses exclusively on discrimination within the pathological subset, which makes it a conditional classifier operating downstream of the binary health-status gate.

# Stage 2 dataset loading

This section loads the rebuilt disease-only hierarchical splits:

- `train_disease.csv`
- `val_disease.csv`
- `test_disease.csv`

These splits are used as the official data source for the Stage 2 disease-specific classifier.

In [67]:
DISEASE_TRAIN_CSV = HIERARCHICAL_SPLITS_DIR / "train_disease.csv"
DISEASE_VAL_CSV = HIERARCHICAL_SPLITS_DIR / "val_disease.csv"
DISEASE_TEST_CSV = HIERARCHICAL_SPLITS_DIR / "test_disease.csv"

stage2_train_df = pd.read_csv(DISEASE_TRAIN_CSV).copy()
stage2_val_df = pd.read_csv(DISEASE_VAL_CSV).copy()
stage2_test_df = pd.read_csv(DISEASE_TEST_CSV).copy()

print("Stage 2 train size:", len(stage2_train_df))
print(stage2_train_df["label"].value_counts())
print()

print("Stage 2 val size:", len(stage2_val_df))
print(stage2_val_df["label"].value_counts())
print()

print("Stage 2 test size:", len(stage2_test_df))
print(stage2_test_df["label"].value_counts())

Stage 2 train size: 4638
label
salmonella     2106
coccidiosis    2071
newcastle       461
Name: count, dtype: int64

Stage 2 val size: 994
label
salmonella     451
coccidiosis    444
newcastle       99
Name: count, dtype: int64

Stage 2 test size: 994
label
salmonella     452
coccidiosis    443
newcastle       99
Name: count, dtype: int64


# Stage 2 path reconstruction

As in the rest of the project, image paths are reconstructed using `filename` rather than `image_path`.

In [68]:
def build_stage2_path(filename: str) -> Path:
    return FINAL_IMAGES_DIR / filename

for df in [stage2_train_df, stage2_val_df, stage2_test_df]:
    df["filepath"] = df["filename"].apply(build_stage2_path)
    df["exists"] = df["filepath"].apply(Path.exists)

print("Stage 2 train exists:")
print(stage2_train_df["exists"].value_counts())
print()

print("Stage 2 val exists:")
print(stage2_val_df["exists"].value_counts())
print()

print("Stage 2 test exists:")
print(stage2_test_df["exists"].value_counts())

if not stage2_train_df["exists"].all():
    raise FileNotFoundError("Some Stage 2 train images were not found.")
if not stage2_val_df["exists"].all():
    raise FileNotFoundError("Some Stage 2 validation images were not found.")
if not stage2_test_df["exists"].all():
    raise FileNotFoundError("Some Stage 2 test images were not found.")

Stage 2 train exists:
exists
True    4638
Name: count, dtype: int64

Stage 2 val exists:
exists
True    994
Name: count, dtype: int64

Stage 2 test exists:
exists
True    994
Name: count, dtype: int64


# Stage 2 label encoding

The multiclass label encoding for the disease-specific classifier is defined as follows:

- `coccidiosis` -> `0`
- `newcastle` -> `1`
- `salmonella` -> `2`

In [69]:
stage2_class_to_idx = {
    "coccidiosis": 0,
    "newcastle": 1,
    "salmonella": 2
}

stage2_idx_to_class = {v: k for k, v in stage2_class_to_idx.items()}

for df in [stage2_train_df, stage2_val_df, stage2_test_df]:
    df["label_idx"] = df["label"].map(stage2_class_to_idx)

print("Stage 2 label encoding completed.")
print(stage2_train_df[["filename", "label", "label_idx"]].head())

Stage 2 label encoding completed.
        filename        label  label_idx
0  CFD_06022.jpg    newcastle          1
1  CFD_01975.jpg  coccidiosis          0
2  CFD_06896.jpg   salmonella          2
3  CFD_06784.jpg   salmonella          2
4  CFD_06940.jpg   salmonella          2


# Stage 2 preprocessing

The disease-specific classifier uses the same image resolution as the rest of the pipeline.

Training uses lightweight augmentation, while validation and test use deterministic preprocessing.

In [70]:
stage2_train_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor()
])

stage2_eval_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor()
])

# Stage 2 dataset class

A dedicated PyTorch dataset is defined for the disease-specific classification stage.

Each sample returns:

- image tensor
- numeric label
- filename
- resolved filepath

In [71]:
class Stage2Dataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None):
        self.df = df.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = Path(row["filepath"])

        try:
            image = Image.open(image_path).convert("RGB")
        except Exception as e:
            raise RuntimeError(f"Error opening Stage 2 image: {image_path} | {e}")

        label = int(row["label_idx"])

        if self.transform is not None:
            image = self.transform(image)

        return image, label, row["filename"], str(image_path)

# Stage 2 datasets and dataloaders

The train, validation, and test partitions are instantiated as PyTorch datasets and wrapped into dataloaders.

In [72]:
stage2_train_dataset = Stage2Dataset(stage2_train_df, transform=stage2_train_transform)
stage2_val_dataset = Stage2Dataset(stage2_val_df, transform=stage2_eval_transform)
stage2_test_dataset = Stage2Dataset(stage2_test_df, transform=stage2_eval_transform)

stage2_train_loader = DataLoader(stage2_train_dataset, batch_size=32, shuffle=True)
stage2_val_loader = DataLoader(stage2_val_dataset, batch_size=32, shuffle=False)
stage2_test_loader = DataLoader(stage2_test_dataset, batch_size=32, shuffle=False)

print("Stage 2 loaders created successfully.")
print("Train batches:", len(stage2_train_loader))
print("Val batches:  ", len(stage2_val_loader))
print("Test batches: ", len(stage2_test_loader))

Stage 2 loaders created successfully.
Train batches: 145
Val batches:   32
Test batches:  32


# Stage 2 model definition

The disease-specific classifier is implemented with a pretrained ResNet-18 backbone.

The final fully connected layer is replaced with a three-output linear classifier corresponding to the three disease classes.

In [73]:
stage2_model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
stage2_in_features = stage2_model.fc.in_features
stage2_model.fc = nn.Linear(stage2_in_features, 3)
stage2_model = stage2_model.to(device)

stage2_criterion = nn.CrossEntropyLoss()
stage2_optimizer = torch.optim.Adam(stage2_model.parameters(), lr=1e-4)

print(stage2_model.fc)

Linear(in_features=512, out_features=3, bias=True)


# Stage 2 training and evaluation utilities

This section defines helper functions for:

- one training epoch,
- full-dataset prediction,
- metric computation,
- checkpoint selection based on validation macro F1.

In [74]:
def train_one_epoch_stage2(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    y_true = []
    y_pred = []

    for images, labels, _, _ in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(logits, dim=1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(y_true, y_pred)
    epoch_macro_f1 = f1_score(y_true, y_pred, average="macro")

    return epoch_loss, epoch_acc, epoch_macro_f1


@torch.no_grad()
def predict_stage2(model, loader, device):
    model.eval()
    rows = []

    for images, labels, filenames, filepaths in loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        for i in range(len(filenames)):
            true_idx = int(labels[i].cpu().item())
            pred_idx = int(preds[i].cpu().item())

            rows.append({
                "filename": filenames[i],
                "filepath": filepaths[i],
                "true_label": stage2_idx_to_class[true_idx],
                "pred_label": stage2_idx_to_class[pred_idx],
                "correct": true_idx == pred_idx,
                "confidence": float(probs[i].max().cpu().item()),
                "prob_coccidiosis": float(probs[i][stage2_class_to_idx["coccidiosis"]].cpu().item()),
                "prob_newcastle": float(probs[i][stage2_class_to_idx["newcastle"]].cpu().item()),
                "prob_salmonella": float(probs[i][stage2_class_to_idx["salmonella"]].cpu().item())
            })

    return pd.DataFrame(rows)


def evaluate_stage2(model, loader, device):
    results_df = predict_stage2(model, loader, device=device)

    y_true = results_df["true_label"]
    y_pred = results_df["pred_label"]

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "confusion_matrix": confusion_matrix(
            y_true,
            y_pred,
            labels=["coccidiosis", "newcastle", "salmonella"]
        ),
        "classification_report": classification_report(
            y_true,
            y_pred,
            labels=["coccidiosis", "newcastle", "salmonella"],
            zero_division=0
        )
    }

    return results_df, metrics

# Stage 2 output paths

Dedicated output paths are defined for:

- best checkpoint,
- training history,
- validation predictions,
- test predictions.

In [75]:
STAGE2_MODEL_PATH = STAGE2_OUTPUT_DIR / "best_stage2_disease_classifier_resnet18.pt"
STAGE2_HISTORY_PATH = STAGE2_OUTPUT_DIR / "stage2_training_history.csv"
STAGE2_VAL_PREDS_PATH = STAGE2_OUTPUT_DIR / "stage2_val_predictions.csv"
STAGE2_TEST_PREDS_PATH = STAGE2_OUTPUT_DIR / "stage2_test_predictions.csv"

print("Stage 2 model path:", STAGE2_MODEL_PATH)

Stage 2 model path: C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\PII\chicken_feces_dataset\03_final\outputs_final_pipeline\stage2_disease_classifier\best_stage2_disease_classifier_resnet18.pt


# Stage 2 training

The Stage 2 disease-specific classifier is trained for a fixed number of epochs.

The best checkpoint is selected using validation macro F1.

In [76]:
STAGE2_NUM_EPOCHS = 12

stage2_history = []
best_stage2_val_macro_f1 = -1.0
best_stage2_epoch = None

for epoch in range(1, STAGE2_NUM_EPOCHS + 1):
    train_loss, train_acc, train_macro_f1 = train_one_epoch_stage2(
        model=stage2_model,
        loader=stage2_train_loader,
        criterion=stage2_criterion,
        optimizer=stage2_optimizer,
        device=device
    )

    _, val_metrics = evaluate_stage2(stage2_model, stage2_val_loader, device=device)

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "train_macro_f1": train_macro_f1,
        "val_acc": val_metrics["accuracy"],
        "val_macro_f1": val_metrics["macro_f1"]
    }
    stage2_history.append(row)

    if val_metrics["macro_f1"] > best_stage2_val_macro_f1:
        best_stage2_val_macro_f1 = val_metrics["macro_f1"]
        best_stage2_epoch = epoch
        torch.save(stage2_model.state_dict(), STAGE2_MODEL_PATH)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} | "
        f"train_acc={train_acc:.4f} | "
        f"train_macro_f1={train_macro_f1:.4f} | "
        f"val_acc={val_metrics['accuracy']:.4f} | "
        f"val_macro_f1={val_metrics['macro_f1']:.4f}"
    )

print("\nBest Stage 2 epoch:", best_stage2_epoch)
print("Best Stage 2 val_macro_f1:", round(best_stage2_val_macro_f1, 4))
print("Best checkpoint saved to:", STAGE2_MODEL_PATH)

Epoch 01 | train_loss=0.1315 | train_acc=0.9506 | train_macro_f1=0.9246 | val_acc=0.9859 | val_macro_f1=0.9768
Epoch 02 | train_loss=0.0309 | train_acc=0.9897 | train_macro_f1=0.9858 | val_acc=0.9879 | val_macro_f1=0.9792
Epoch 03 | train_loss=0.0295 | train_acc=0.9907 | train_macro_f1=0.9869 | val_acc=0.9899 | val_macro_f1=0.9861
Epoch 04 | train_loss=0.0159 | train_acc=0.9944 | train_macro_f1=0.9916 | val_acc=0.9960 | val_macro_f1=0.9944
Epoch 05 | train_loss=0.0092 | train_acc=0.9972 | train_macro_f1=0.9968 | val_acc=0.9920 | val_macro_f1=0.9865
Epoch 06 | train_loss=0.0110 | train_acc=0.9963 | train_macro_f1=0.9936 | val_acc=0.9899 | val_macro_f1=0.9859
Epoch 07 | train_loss=0.0137 | train_acc=0.9955 | train_macro_f1=0.9930 | val_acc=0.9920 | val_macro_f1=0.9874
Epoch 08 | train_loss=0.0101 | train_acc=0.9968 | train_macro_f1=0.9945 | val_acc=0.9960 | val_macro_f1=0.9944
Epoch 09 | train_loss=0.0085 | train_acc=0.9968 | train_macro_f1=0.9951 | val_acc=0.9950 | val_macro_f1=0.9937
E

# Stage 2 training history

The epoch-level training history is stored as a dataframe for inspection and export.

In [77]:
stage2_history_df = pd.DataFrame(stage2_history)
stage2_history_df.to_csv(STAGE2_HISTORY_PATH, index=False)

display(stage2_history_df)
print("Training history saved to:", STAGE2_HISTORY_PATH)

,epoch,train_loss,train_acc,train_macro_f1,val_acc,val_macro_f1
0,1,0.131506,0.950625,0.924638,0.985915,0.976786
1,2,0.030877,0.989651,0.985804,0.987928,0.979199
2,3,0.029451,0.990729,0.986935,0.989940,0.986121
3,4,0.015930,0.994394,0.991611,0.995976,0.994436
4,5,0.009243,0.997197,0.996801,0.991952,0.986474
5,6,0.011010,0.996335,0.993628,0.989940,0.985875
6,7,0.013713,0.995472,0.992984,0.991952,0.987373
7,8,0.010083,0.996766,0.994514,0.995976,0.994434
8,9,0.008506,0.996766,0.995066,0.994970,0.993661
9,10,0.006425,0.997844,0.997000,0.996982,0.995148


Training history saved to: C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\PII\chicken_feces_dataset\03_final\outputs_final_pipeline\stage2_disease_classifier\stage2_training_history.csv


# Stage 2 best-checkpoint loading

The best validation checkpoint is reloaded before final validation and test evaluation.

In [78]:
best_stage2_model = models.resnet18(weights=None)
best_stage2_model.fc = nn.Linear(best_stage2_model.fc.in_features, 3)
best_stage2_model.load_state_dict(torch.load(STAGE2_MODEL_PATH, map_location=device))
best_stage2_model = best_stage2_model.to(device)
best_stage2_model.eval()

print("Best Stage 2 model loaded successfully.")

Best Stage 2 model loaded successfully.


# Stage 2 validation evaluation

The best checkpoint is evaluated on the validation split.

In [79]:
stage2_val_results, stage2_val_metrics = evaluate_stage2(
    best_stage2_model,
    stage2_val_loader,
    device=device
)

stage2_val_results.to_csv(STAGE2_VAL_PREDS_PATH, index=False)

display(stage2_val_results.head())
print("Stage 2 validation accuracy:", round(stage2_val_metrics["accuracy"], 4))
print("Stage 2 validation macro F1:", round(stage2_val_metrics["macro_f1"], 4))
print("\nStage 2 validation confusion matrix:")
print(stage2_val_metrics["confusion_matrix"])
print("\nStage 2 validation classification report:")
print(stage2_val_metrics["classification_report"])

,filename,filepath,true_label,pred_label,correct,confidence,prob_coccidiosis,prob_newcastle,prob_salmonella
0,CFD_02458.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,coccidiosis,coccidiosis,True,0.999960,9.999603e-01,0.000040,1.683605e-07
1,CFD_06694.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,salmonella,salmonella,True,0.999997,4.084410e-09,0.000003,9.999971e-01
2,CFD_09550.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,salmonella,salmonella,True,0.999996,1.487763e-08,0.000004,9.999963e-01
3,CFD_06501.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,newcastle,newcastle,True,0.999605,1.965864e-04,0.999605,1.980848e-04
4,CFD_00912.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,coccidiosis,coccidiosis,True,0.999683,9.996831e-01,0.000075,2.418690e-04


Stage 2 validation accuracy: 0.997
Stage 2 validation macro F1: 0.9951

Stage 2 validation confusion matrix:
[[444   0   0]
 [  0  98   1]
 [  1   1 449]]

Stage 2 validation classification report:
              precision    recall  f1-score   support

 coccidiosis       1.00      1.00      1.00       444
   newcastle       0.99      0.99      0.99        99
  salmonella       1.00      1.00      1.00       451

    accuracy                           1.00       994
   macro avg       1.00      1.00      1.00       994
weighted avg       1.00      1.00      1.00       994



# Stage 2 test evaluation

The best checkpoint is evaluated on the held-out test split.

In [80]:
stage2_test_results, stage2_test_metrics = evaluate_stage2(
    best_stage2_model,
    stage2_test_loader,
    device=device
)

stage2_test_results.to_csv(STAGE2_TEST_PREDS_PATH, index=False)

display(stage2_test_results.head())
print("Stage 2 test accuracy:", round(stage2_test_metrics["accuracy"], 4))
print("Stage 2 test macro F1:", round(stage2_test_metrics["macro_f1"], 4))
print("\nStage 2 test confusion matrix:")
print(stage2_test_metrics["confusion_matrix"])
print("\nStage 2 test classification report:")
print(stage2_test_metrics["classification_report"])

print("\nValidation predictions saved to:", STAGE2_VAL_PREDS_PATH)
print("Test predictions saved to:", STAGE2_TEST_PREDS_PATH)

,filename,filepath,true_label,pred_label,correct,confidence,prob_coccidiosis,prob_newcastle,prob_salmonella
0,CFD_08622.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,salmonella,salmonella,True,0.999999,3.748001e-09,7.208048e-07,9.999993e-01
1,CFD_01667.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,coccidiosis,coccidiosis,True,0.999997,9.999968e-01,2.032216e-06,1.205127e-06
2,CFD_08158.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,salmonella,salmonella,True,1.000000,1.000047e-13,5.541579e-09,1.000000e+00
3,CFD_07551.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,salmonella,salmonella,True,0.999904,1.088044e-08,9.612974e-05,9.999039e-01
4,CFD_00743.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,coccidiosis,coccidiosis,True,0.999996,9.999957e-01,4.120348e-06,7.433876e-08


Stage 2 test accuracy: 0.994
Stage 2 test macro F1: 0.9889

Stage 2 test confusion matrix:
[[441   1   1]
 [  3  96   0]
 [  0   1 451]]

Stage 2 test classification report:
              precision    recall  f1-score   support

 coccidiosis       0.99      1.00      0.99       443
   newcastle       0.98      0.97      0.97        99
  salmonella       1.00      1.00      1.00       452

    accuracy                           0.99       994
   macro avg       0.99      0.99      0.99       994
weighted avg       0.99      0.99      0.99       994


Validation predictions saved to: C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\PII\chicken_feces_dataset\03_final\outputs_final_pipeline\stage2_disease_classifier\stage2_val_predictions.csv
Test predictions saved to: C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\PII\chicken_feces_dataset\03_final\outputs_final_pipeline\stage2_disease_classifier\stage2_test_predictions.csv


# Stage 2 interpretation checkpoint

At this point, the disease-specific classifier has been trained and evaluated on validation and test data.

Before proceeding to the final hierarchical integration step, the Stage 2 results should be reviewed in order to assess whether disease discrimination within the pathological subset is sufficiently strong and stable.

# Stage 2 results and interpretation

The Stage 2 disease-specific classifier showed excellent internal performance on the rebuilt disease-only hierarchical splits.

## Summary of results

The best checkpoint was selected at **epoch 10** based on validation macro F1.

### Validation performance
- **accuracy = 0.9970**
- **macro F1 = 0.9951**

### Test performance
- **accuracy = 0.9940**
- **macro F1 = 0.9889**

## Confusion-matrix interpretation

### Validation confusion matrix
- `coccidiosis`: 444 correctly classified, 0 errors
- `newcastle`: 98 correctly classified, 1 error
- `salmonella`: 449 correctly classified, 2 errors

### Test confusion matrix
- `coccidiosis`: 441 correctly classified, 2 errors
- `newcastle`: 96 correctly classified, 3 errors
- `salmonella`: 451 correctly classified, 1 error

These results indicate that disease discrimination within the pathological subset is highly reliable in the rebuilt internal setting.

## Interpretation

The Stage 2 classifier is a strong component of the hierarchical pipeline. Once a sample has been routed into the pathological branch, the model is able to distinguish between `coccidiosis`, `newcastle`, and `salmonella` with very high accuracy and macro F1.

Among the three diseases, `newcastle` remains the most delicate class, but its performance is still strong and does not compromise the overall quality of the stage.

## Methodological conclusion

The Stage 2 results support a key conclusion of the project: the main bottleneck is not disease discrimination within the pathological subset, but the earlier stages of the pipeline, especially those related to input validity and the healthy-versus-unhealthy decision boundary.

# Final hierarchical pipeline definition

Based on the experimental results obtained in Stages 0, 1, and 2, the final operational hierarchical pipeline is defined as follows:

1. **Stage 1**: `healthy` vs `unhealthy`
2. if `unhealthy`, **Stage 2**: `coccidiosis` vs `newcastle` vs `salmonella`

## Note on Stage 0

Although a Stage 0 feces-presence gate was implemented and showed excellent internal performance on its auxiliary dataset, its screening behavior on the large feces dataset produced a false rejection rate that was too high for hard integration into the final operational pipeline.

Therefore, Stage 0 is documented as an exploratory auxiliary module, but it is not used as a strict rejection gate in the final hierarchical inference chain.

# Final hierarchical inference logic

The final operational pipeline applies the following logic:

- if Stage 1 predicts `healthy`, the final output is `healthy`
- if Stage 1 predicts `unhealthy`, the sample is forwarded to Stage 2
- Stage 2 then produces one of:
  - `coccidiosis`
  - `newcastle`
  - `salmonella`

In [82]:
@torch.no_grad()
def predict_stage1_single(image_path, model, transform, device):
    model.eval()

    image = Image.open(image_path).convert("RGB")
    x = transform(image).unsqueeze(0).to(device)

    logits = model(x)
    probs = torch.softmax(logits, dim=1)[0]
    pred_idx = int(torch.argmax(probs).cpu().item())

    return {
        "pred_label": stage1_idx_to_class[pred_idx],
        "confidence": float(probs[pred_idx].cpu().item()),
        "prob_healthy": float(probs[stage1_class_to_idx["healthy"]].cpu().item()),
        "prob_unhealthy": float(probs[stage1_class_to_idx["unhealthy"]].cpu().item())
    }


@torch.no_grad()
def predict_stage2_single(image_path, model, transform, device):
    model.eval()

    image = Image.open(image_path).convert("RGB")
    x = transform(image).unsqueeze(0).to(device)

    logits = model(x)
    probs = torch.softmax(logits, dim=1)[0]
    pred_idx = int(torch.argmax(probs).cpu().item())

    return {
        "pred_label": stage2_idx_to_class[pred_idx],
        "confidence": float(probs[pred_idx].cpu().item()),
        "prob_coccidiosis": float(probs[stage2_class_to_idx["coccidiosis"]].cpu().item()),
        "prob_newcastle": float(probs[stage2_class_to_idx["newcastle"]].cpu().item()),
        "prob_salmonella": float(probs[stage2_class_to_idx["salmonella"]].cpu().item())
    }


@torch.no_grad()
def predict_final_pipeline_single(image_path, stage1_model, stage2_model, transform, device):
    stage1_out = predict_stage1_single(image_path, stage1_model, transform, device)

    if stage1_out["pred_label"] == "healthy":
        return {
            "final_pred_label": "healthy",
            "stage1_pred_label": stage1_out["pred_label"],
            "stage1_confidence": stage1_out["confidence"],
            "stage2_pred_label": None,
            "stage2_confidence": None
        }

    stage2_out = predict_stage2_single(image_path, stage2_model, transform, device)

    return {
        "final_pred_label": stage2_out["pred_label"],
        "stage1_pred_label": stage1_out["pred_label"],
        "stage1_confidence": stage1_out["confidence"],
        "stage2_pred_label": stage2_out["pred_label"],
        "stage2_confidence": stage2_out["confidence"]
    }

# Pipeline-level internal evaluation dataset

The final hierarchical pipeline is evaluated internally on the rebuilt multiclass test split.

This evaluation is particularly important because it reflects the complete end-to-end behavior of the final operational cascade.

In [83]:
final_pipeline_test_df = pd.read_csv(MULTICLASS_TEST_CSV).copy()
final_pipeline_test_df["filepath"] = final_pipeline_test_df["filename"].apply(build_large_feces_path)
final_pipeline_test_df["exists"] = final_pipeline_test_df["filepath"].apply(Path.exists)

print("Final pipeline test size:", len(final_pipeline_test_df))
print(final_pipeline_test_df["label"].value_counts())
print(final_pipeline_test_df["exists"].value_counts())

if not final_pipeline_test_df["exists"].all():
    raise FileNotFoundError("Some files in the final pipeline test set were not found.")

Final pipeline test size: 1444
label
salmonella     452
healthy        450
coccidiosis    443
newcastle       99
Name: count, dtype: int64
exists
True    1444
Name: count, dtype: int64


# Final hierarchical pipeline evaluation function

A dataframe-based evaluation function is defined for the complete Stage 1 -> Stage 2 hierarchical cascade.

In [84]:
@torch.no_grad()
def evaluate_final_pipeline(df, stage1_model, stage2_model, transform, device):
    rows = []

    for _, row in df.iterrows():
        image_path = row["filepath"]
        true_label = row["label"]

        pred = predict_final_pipeline_single(
            image_path=image_path,
            stage1_model=stage1_model,
            stage2_model=stage2_model,
            transform=transform,
            device=device
        )

        rows.append({
            "filename": row["filename"],
            "filepath": str(image_path),
            "true_label": true_label,
            "final_pred_label": pred["final_pred_label"],
            "stage1_pred_label": pred["stage1_pred_label"],
            "stage1_confidence": pred["stage1_confidence"],
            "stage2_pred_label": pred["stage2_pred_label"],
            "stage2_confidence": pred["stage2_confidence"],
            "correct": true_label == pred["final_pred_label"]
        })

    results_df = pd.DataFrame(rows)

    y_true = results_df["true_label"]
    y_pred = results_df["final_pred_label"]

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            labels=["healthy", "coccidiosis", "newcastle", "salmonella"],
            average="macro"
        ),
        "confusion_matrix": confusion_matrix(
            y_true,
            y_pred,
            labels=["healthy", "coccidiosis", "newcastle", "salmonella"]
        ),
        "classification_report": classification_report(
            y_true,
            y_pred,
            labels=["healthy", "coccidiosis", "newcastle", "salmonella"],
            zero_division=0
        )
    }

    return results_df, metrics

# Final hierarchical pipeline internal evaluation

The complete operational cascade is now evaluated on the rebuilt multiclass test split.

In [85]:
PIPELINE_TEST_PREDS_PATH = PIPELINE_OUTPUT_DIR / "final_pipeline_test_predictions.csv"

final_pipeline_test_results, final_pipeline_test_metrics = evaluate_final_pipeline(
    final_pipeline_test_df,
    best_stage1_model,
    best_stage2_model,
    stage1_eval_transform,
    device=device
)

final_pipeline_test_results.to_csv(PIPELINE_TEST_PREDS_PATH, index=False)

display(final_pipeline_test_results.head())
print("Final pipeline test accuracy:", round(final_pipeline_test_metrics["accuracy"], 4))
print("Final pipeline test macro F1:", round(final_pipeline_test_metrics["macro_f1"], 4))
print("\nFinal pipeline test confusion matrix:")
print(final_pipeline_test_metrics["confusion_matrix"])
print("\nFinal pipeline test classification report:")
print(final_pipeline_test_metrics["classification_report"])
print("\nSaved to:", PIPELINE_TEST_PREDS_PATH)

,filename,filepath,true_label,final_pred_label,stage1_pred_label,stage1_confidence,stage2_pred_label,stage2_confidence,correct
0,CFD_04544.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,healthy,healthy,healthy,1.000000,None,NaN,True
1,CFD_08622.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,salmonella,salmonella,unhealthy,1.000000,salmonella,0.999999,True
2,CFD_01667.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,coccidiosis,coccidiosis,unhealthy,0.999926,coccidiosis,0.999997,True
3,CFD_08158.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,salmonella,salmonella,unhealthy,0.999995,salmonella,1.000000,True
4,CFD_07551.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,salmonella,salmonella,unhealthy,0.999984,salmonella,0.999904,True


Final pipeline test accuracy: 0.9751
Final pipeline test macro F1: 0.9708

Final pipeline test confusion matrix:
[[437   3   1   9]
 [  7 434   1   1]
 [  4   3  92   0]
 [  7   0   0 445]]

Final pipeline test classification report:
              precision    recall  f1-score   support

     healthy       0.96      0.97      0.97       450
 coccidiosis       0.99      0.98      0.98       443
   newcastle       0.98      0.93      0.95        99
  salmonella       0.98      0.98      0.98       452

    accuracy                           0.98      1444
   macro avg       0.98      0.97      0.97      1444
weighted avg       0.98      0.98      0.98      1444


Saved to: C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\PII\chicken_feces_dataset\03_final\outputs_final_pipeline\hierarchical_pipeline\final_pipeline_test_predictions.csv


# External benchmark loading

The manually curated external benchmark is loaded as a separate evaluation set.

This benchmark is frozen and is used to assess external behavior on a small but clinically relevant set of difficult examples.

In [86]:
EXTERNAL_BENCHMARK_DIR = ROOT / "01_raw" / "external_quantitative" / "images"

external_benchmark_rows = [
    {"filename": "healthy_fed_green_01.jpg", "label": "healthy"},
    {"filename": "healthy_fed_yellow_01.jpg", "label": "healthy"},
    {"filename": "healthy_fed_cecal_01.jpg", "label": "healthy"},
    {"filename": "healthy_cj_regular_01.jpg", "label": "healthy"},
    {"filename": "healthy_cj_green_01.jpg", "label": "healthy"},
    {"filename": "healthy_cj_yellow_01.jpg", "label": "healthy"},
    {"filename": "healthy_buffclucks_cecal_01.png", "label": "healthy"},
    {"filename": "coccidiosis_fed_confirmed_01.png", "label": "coccidiosis"},
    {"filename": "coccidiosis_cornell_bloody_01.jpeg", "label": "coccidiosis"},
    {"filename": "coccidiosis_cornell_bloody_02.jpeg", "label": "coccidiosis"},
]

external_benchmark_df = pd.DataFrame(external_benchmark_rows)
external_benchmark_df["filepath"] = external_benchmark_df["filename"].apply(lambda x: EXTERNAL_BENCHMARK_DIR / x)
external_benchmark_df["exists"] = external_benchmark_df["filepath"].apply(Path.exists)

print("External benchmark size:", len(external_benchmark_df))
print(external_benchmark_df["label"].value_counts())
print(external_benchmark_df["exists"].value_counts())

if not external_benchmark_df["exists"].all():
    missing_external = external_benchmark_df.loc[~external_benchmark_df["exists"], ["filename", "filepath"]]
    display(missing_external)
    raise FileNotFoundError("Some external benchmark files were not found.")

External benchmark size: 10
label
healthy        7
coccidiosis    3
Name: count, dtype: int64
exists
True    10
Name: count, dtype: int64


# Final hierarchical pipeline external evaluation

The complete operational cascade is now evaluated on the frozen external benchmark.

In [88]:
PIPELINE_EXTERNAL_PREDS_PATH = PIPELINE_OUTPUT_DIR / "final_pipeline_external_predictions.csv"

final_pipeline_external_results, final_pipeline_external_metrics = evaluate_final_pipeline(
    external_benchmark_df,
    best_stage1_model,
    best_stage2_model,
    stage1_eval_transform,
    device=device
)

final_pipeline_external_results.to_csv(PIPELINE_EXTERNAL_PREDS_PATH, index=False)

display(final_pipeline_external_results)
print("Final pipeline external accuracy:", round(final_pipeline_external_metrics["accuracy"], 4))
print("Final pipeline external macro F1:", round(final_pipeline_external_metrics["macro_f1"], 4))
print("\nFinal pipeline external confusion matrix:")
print(final_pipeline_external_metrics["confusion_matrix"])
print("\nFinal pipeline external classification report:")
print(final_pipeline_external_metrics["classification_report"])
print("\nSaved to:", PIPELINE_EXTERNAL_PREDS_PATH)

C:\Users\usuario\miniconda3\envs\chicken_feces\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


,filename,filepath,true_label,final_pred_label,stage1_pred_label,stage1_confidence,stage2_pred_label,stage2_confidence,correct
0,healthy_fed_green_01.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,healthy,newcastle,unhealthy,0.727527,newcastle,0.968211,False
1,healthy_fed_yellow_01.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,healthy,healthy,healthy,0.573222,None,NaN,True
2,healthy_fed_cecal_01.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,healthy,coccidiosis,unhealthy,0.738828,coccidiosis,0.999972,False
3,healthy_cj_regular_01.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,healthy,healthy,healthy,0.965841,None,NaN,True
4,healthy_cj_green_01.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,healthy,coccidiosis,unhealthy,0.963138,coccidiosis,0.999854,False
5,healthy_cj_yellow_01.jpg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,healthy,coccidiosis,unhealthy,0.976977,coccidiosis,0.999863,False
6,healthy_buffclucks_cecal_01.png,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,healthy,coccidiosis,unhealthy,0.983926,coccidiosis,0.999986,False
7,coccidiosis_fed_confirmed_01.png,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,coccidiosis,coccidiosis,unhealthy,0.940077,coccidiosis,0.973485,True
8,coccidiosis_cornell_bloody_01.jpeg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,coccidiosis,healthy,healthy,0.568052,None,NaN,False
9,coccidiosis_cornell_bloody_02.jpeg,C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\...,coccidiosis,coccidiosis,unhealthy,0.967028,coccidiosis,0.984320,True


Final pipeline external accuracy: 0.4
Final pipeline external macro F1: 0.2111

Final pipeline external confusion matrix:
[[2 4 1 0]
 [1 2 0 0]
 [0 0 0 0]
 [0 0 0 0]]

Final pipeline external classification report:
              precision    recall  f1-score   support

     healthy       0.67      0.29      0.40         7
 coccidiosis       0.33      0.67      0.44         3
   newcastle       0.00      0.00      0.00         0
  salmonella       0.00      0.00      0.00         0

    accuracy                           0.40        10
   macro avg       0.25      0.24      0.21        10
weighted avg       0.57      0.40      0.41        10


Saved to: C:\Users\usuario\Desktop\IA\3º\Segundo cuatri\PII\chicken_feces_dataset\03_final\outputs_final_pipeline\hierarchical_pipeline\final_pipeline_external_predictions.csv


# Final pipeline interpretation checkpoint

At this point, the final operational hierarchical pipeline has been evaluated both internally and externally.

These results should be reviewed before writing the final discussion, because they determine the practical interpretation of the full system:
- strong internal performance,
- external benchmark behavior,
- and the relationship between the final pipeline and the broader project diagnosis.

# Final pipeline results and interpretation

The final operational hierarchical pipeline was evaluated both internally on the rebuilt multiclass test split and externally on the frozen manually curated benchmark.

## Internal evaluation

The final Stage 1 -> Stage 2 cascade achieved strong internal performance on the rebuilt multiclass test split:

- **accuracy = 0.9751**
- **macro F1 = 0.9708**

### Internal confusion-matrix summary
- `healthy`: 437 correctly classified out of 450
- `coccidiosis`: 434 correctly classified out of 443
- `newcastle`: 92 correctly classified out of 99
- `salmonella`: 445 correctly classified out of 452

These results indicate that the final operational pipeline is internally stable and effective within the reconstructed dataset setting.

## External evaluation

On the frozen external benchmark, the same pipeline achieved:

- **accuracy = 0.4000**
- **macro F1 = 0.2111**

The external prediction pattern reproduced the same generalization issues identified earlier in the project:

- difficult `healthy` samples were frequently overdiagnosed as pathological,
- normal green variants again triggered spurious `newcastle` predictions,
- cecal and yellow normal variants were often pushed toward `coccidiosis`,
- one real `coccidiosis` case was incorrectly filtered as `healthy`.

## Interpretation

The final notebook confirms a central project conclusion:

the hierarchical pipeline can achieve very high internal performance, but external generalization is still limited by the representation of difficult normal fecal patterns.

In other words, the main bottleneck is not the discrimination between diseases once a sample has already entered the pathological branch. Instead, the critical limitation remains the visual boundary between `healthy` and abnormal-looking but still normal fecal variants.

## Final methodological conclusion

The final operational pipeline should therefore be interpreted as:

- a strong and well-implemented internal baseline,
- a technically consistent hierarchical system,
- and a useful experimental reconstruction that confirms the real generalization bottleneck of the project.

Its main limitation is not internal optimization, but the insufficient coverage of the visual variability of `healthy`, especially in borderline external cases.

# Final comparison across pipeline formulations

The project explored multiple formulations of the poultry feces classification task, moving from a direct multiclass baseline to a rebuilt hierarchical pipeline.

The table below summarizes the most relevant final results across the main evaluated configurations, with special emphasis on the distinction between:

- internal performance,
- external benchmark behavior,
- and the practical interpretation of each pipeline.

In [89]:
final_comparison_rows = [
    {
        "pipeline": "Original multiclass baseline",
        "internal_setting": "4-class direct classifier",
        "internal_result": "Historical baseline",
        "external_result": "accuracy = 0.40 | healthy = 2/7 | coccidiosis = 2/3",
        "main_reading": "Functional baseline, but weak external generalization on difficult healthy cases."
    },
    {
        "pipeline": "Multiclass + repair_v1",
        "internal_setting": "4-class direct classifier with added difficult healthy samples",
        "internal_result": "Historical repair stage",
        "external_result": "accuracy = 0.50 | healthy = 4/7 | coccidiosis = 1/3",
        "main_reading": "Improved healthy recall, but pushed the decision boundary too far toward healthy."
    },
    {
        "pipeline": "Binary repair_v2",
        "internal_setting": "healthy vs unhealthy",
        "internal_result": "Historical repair stage",
        "external_result": "accuracy = 0.50 | healthy = 2/7 | unhealthy = 3/3",
        "main_reading": "Strong pathological coverage, but still overly harsh on difficult healthy variants."
    },
    {
        "pipeline": "Stage 0 auxiliary gate",
        "internal_setting": "feces_present vs no_feces",
        "internal_result": "val/test accuracy = 1.0000 | macro F1 = 1.0000",
        "external_result": "screening on large feces dataset: false rejection rate = 0.1070",
        "main_reading": "Promising auxiliary module, but not robust enough for hard integration."
    },
    {
        "pipeline": "Stage 1 binary health classifier",
        "internal_setting": "healthy vs unhealthy",
        "internal_result": "test accuracy = 0.9785 | macro F1 = 0.9751",
        "external_result": "Evaluated within final cascade",
        "main_reading": "Strong and stable internally; healthy remains the more delicate class."
    },
    {
        "pipeline": "Stage 2 disease classifier",
        "internal_setting": "coccidiosis vs newcastle vs salmonella",
        "internal_result": "test accuracy = 0.9940 | macro F1 = 0.9889",
        "external_result": "Evaluated within final cascade",
        "main_reading": "Very strong internal disease discrimination within the pathological subset."
    },
    {
        "pipeline": "Final operational hierarchical pipeline",
        "internal_setting": "Stage 1 -> Stage 2",
        "internal_result": "test accuracy = 0.9751 | macro F1 = 0.9708",
        "external_result": "accuracy = 0.40 | macro F1 = 0.2111",
        "main_reading": "Internally strong, but externally limited by difficult healthy generalization."
    },
]

final_comparison_df = pd.DataFrame(final_comparison_rows)
display(final_comparison_df)

,pipeline,internal_setting,internal_result,external_result,main_reading
0,Original multiclass baseline,4-class direct classifier,Historical baseline,accuracy = 0.40 | healthy = 2/7 | coccidiosis ...,"Functional baseline, but weak external general..."
1,Multiclass + repair_v1,4-class direct classifier with added difficult...,Historical repair stage,accuracy = 0.50 | healthy = 4/7 | coccidiosis ...,"Improved healthy recall, but pushed the decisi..."
2,Binary repair_v2,healthy vs unhealthy,Historical repair stage,accuracy = 0.50 | healthy = 2/7 | unhealthy = 3/3,"Strong pathological coverage, but still overly..."
3,Stage 0 auxiliary gate,feces_present vs no_feces,val/test accuracy = 1.0000 | macro F1 = 1.0000,screening on large feces dataset: false reject...,"Promising auxiliary module, but not robust eno..."
4,Stage 1 binary health classifier,healthy vs unhealthy,test accuracy = 0.9785 | macro F1 = 0.9751,Evaluated within final cascade,Strong and stable internally; healthy remains ...
5,Stage 2 disease classifier,coccidiosis vs newcastle vs salmonella,test accuracy = 0.9940 | macro F1 = 0.9889,Evaluated within final cascade,Very strong internal disease discrimination wi...
6,Final operational hierarchical pipeline,Stage 1 -> Stage 2,test accuracy = 0.9751 | macro F1 = 0.9708,accuracy = 0.40 | macro F1 = 0.2111,"Internally strong, but externally limited by d..."


# Discussion

This notebook consolidates the final experimental reconstruction of the hierarchical poultry feces classification pipeline.

## Main findings

The results show a clear pattern across all stages of the project:

- internal performance can become very strong,
- disease discrimination within the pathological subset is reliable,
- but the main generalization bottleneck remains unresolved on the external benchmark.

The final operational hierarchical pipeline achieved high internal performance on the rebuilt multiclass test split, which confirms that the system is technically well implemented and that the learned representations are meaningful within the rebuilt data setting.

However, the external benchmark reproduced the same central error pattern observed throughout the project:

- difficult `healthy` samples were frequently overdiagnosed as pathological,
- normal green variants again triggered spurious `newcastle` predictions,
- cecal and yellow normal variants were often pushed toward `coccidiosis`,
- and one real `coccidiosis` case was still filtered as `healthy`.

## Interpretation of the hierarchical formulation

The hierarchical decomposition was methodologically reasonable:

- separating healthy from unhealthy before disease discrimination is conceptually sound,
- and disease discrimination itself turned out to be very strong once the case had already entered the pathological branch.

Nevertheless, the hierarchy did not automatically solve the core problem of the project. Its limiting factor remained the quality of the earlier decision boundary, especially the visual distinction between normal but difficult `healthy` samples and pathological-looking cases.

## Interpretation of Stage 0

The auxiliary Stage 0 module (`feces_present` vs `no_feces`) achieved perfect internal performance on its dedicated auxiliary dataset, but its screening behavior on the large feces dataset revealed a false rejection rate that was too high for hard integration.

This means that Stage 0 is useful as an exploratory auxiliary module, but cannot yet be considered robust enough to act as a strict front-end rejection gate in the final system.

## Overall reading

The project should therefore not be interpreted as a failed attempt at classification, but as a strong baseline and a well-diagnosed experimental system.

The evidence supports the following reading:

- the system learns useful visual signal,
- the internal pipeline is strong,
- the disease classifier is not the main bottleneck,
- and the true difficulty lies in the insufficient representation of the visual variability of `healthy`, especially under external and borderline conditions.

# Conclusions

The final hierarchical reconstruction confirms the central conclusion of the project:

**the main limitation of the system is not internal optimization, but external generalization on difficult healthy samples.**

## Final conclusions

1. A technically consistent and internally strong hierarchical pipeline was successfully built.
2. The binary health-status classifier achieved strong internal performance.
3. The disease-specific classifier achieved very strong internal performance within the pathological subset.
4. The complete operational cascade also performed strongly on the rebuilt internal multiclass test set.
5. Despite this, the frozen external benchmark continued to expose the same core bottleneck:
   - overdiagnosis of difficult `healthy` cases,
   - recurring confusion between normal variants and pathology,
   - and persistence of the `green normal -> newcastle` error pattern.
6. The auxiliary Stage 0 gate was promising internally, but its false rejection rate on valid fecal images was too high for hard deployment in the final cascade.

## Final interpretation

The final outcome of the project is therefore not a failed model, but a well-characterized baseline with a clearly identified generalization bottleneck.

The project demonstrates that:

- high internal metrics are achievable,
- the pipeline design is meaningful,
- and the most important next step is not simply more training, but improving the representation of difficult normal fecal variability without contaminating the dataset.

## Reasonable future directions

The most justified future directions are:

- improving coverage of difficult `healthy` variants,
- performing finer error analysis on externally misclassified healthy images,
- studying the recurring green-to-`newcastle` bias in more detail,
- and refining auxiliary rejection logic only if it can be shown to preserve a very low false rejection rate on valid fecal samples.